# python 미니 프로젝트 - 개인 예산관리 AI Agent

<details>
<summary>설명 펼치기</summary>

---

---

## **개인 예산관리 AI Agent**

---

## * 프로젝트 개요

사용자가 자연어로 수입과 지출을 기록하고 예산을 관리할수있는 ai agent 구현

agent는 사용자의 요청을 이해한뒤 적절한 도구를 선택하여 
- 거래 내역 등록
- 검색
- 수정
- 예산 조회

기능을 수행할수있어야한다.

이 프로젝트의 목적은 단순 금융 계산기가 아니라 gemini의 functon callling 을 이용하여 실제 기능을 수행하는 agent를 만드는것이다.

---

</details>

### 이 프로젝트를 어떤 순서로, 왜 그 순서로 만드는가

<details>
<summary>설명 펼치기</summary>

---

#### 완성된 프로그램의 구조

```
사용자: "오늘 점심으로 12000원 썼어"
   │
   ▼
Gemini                    ← 말을 알아듣고 판단만 한다
   │                         "add_transaction을 이런 인자로 불러줘"
   ▼
도구 함수                  ← 실제로 일한다 (파이썬)
   │
   ▼
BudgetManager 안의 데이터   ← 실제로 보관한다 (거래 목록, 예산)
   │
   ▼
data.json                 ← 껐다 켜도 남는다
```

**Gemini는 아무것도 저장하지 못한다.** 우리 코드를 볼 수도, 실행할 수도 없다.
"이 함수를 이렇게 불러주세요"라고 **요청**할 뿐이고, 실제 데이터와 기능은 전부 파이썬 쪽에 있다.

그래서 이 프로젝트의 **본체는 파이썬 코드**이고, Gemini는 그 앞에 붙는 **말을 알아듣는 창구**다.

#### 만드는 순서

위 그림의 **아래쪽부터** 만든다. 담을 그릇 → 그릇을 다루는 기능 → 그 기능을 Gemini에 연결.

| Phase | 만드는 것 | 왜 이 순서인가 |
|---|---|---|
| **1** | 데이터를 담을 그릇 + 검증 규칙 | 담을 곳이 없으면 "등록하는 기능"을 만들 수가 없다 |
| **2~3** | 그릇을 다루는 기능 5개 | 나중에 Gemini가 부를 도구들. **아직 Gemini는 등장하지 않는다** |
| **4** | 파일에 저장하고 복원하기 | 기능이 완성돼야 무엇을 저장할지 정해진다 |
| **5~6** | Gemini 연결 + 반복 루프 | 이미 동작하는 기능을 도구로 등록만 하면 된다 |
| **7~8** | 시연 검증 + 선택 기능 | 필수가 다 되고 나서 확장한다 |
| **9~10** | `.py`로 옮기고 마무리 | |

#### Gemini를 마지막에 붙이는 이유

명세서에도 못 박혀 있다.

> *"먼저 거래 등록, 검색, 수정과 예산 관리 기능을 Python 함수로 구현한다.
> 각 함수를 직접 호출하여 입력에 맞는 결과가 반환되는지 확인한 뒤 Gemini Function Calling에 도구로 연결한다."*

Gemini부터 붙이면, 에러가 났을 때 **함수가 잘못된 건지 Agent 연결이 잘못된 건지 구분할 수 없다.**
함수가 확실히 동작하는 상태에서 연결하면, 문제가 생겨도 원인이 연결 쪽이라는 걸 알 수 있다.

#### Phase 1~4 동안은 Gemini가 없다

Phase 4까지는 **평범한 파이썬 프로그램**을 만드는 것과 똑같다.
`manager.add_transaction("지출", "식비", 12000, "2026-08-27")` 처럼 **우리가 직접 함수를 부른다.**

Phase 5에서 바뀌는 건 딱 하나다. **함수를 부르는 주체가 "나"에서 "Gemini"로 바뀐다.**
함수 자체는 하나도 고치지 않는다.

---

</details>

### Phase 1 — 데이터 구조와 검증 함수

<details>
<summary>설명 펼치기</summary>

---

#### 1-1. 준비 — 필요한 도구 가져오기

##### `import`는 왜 필요한가

파이썬은 프로그램을 시작할 때 **아주 기본적인 기능만** 들고 있다. `print()`, `len()`, `int()` 같은 것들이다.
날짜를 다루거나 파일을 저장하는 기능은 **표준 라이브러리**라는 창고에 들어 있고, 파이썬은 그걸 미리 꺼내두지 않는다.

`import`는 **그 창고에서 필요한 상자를 꺼내오는 명령**이다.

처음부터 전부 꺼내두지 않는 이유는 두 가지다.
- 안 쓰는 기능까지 메모리에 올리면 무겁고 느려진다.
- 이름이 겹칠 수 있다. (여러 상자에 같은 이름의 기능이 있을 수 있다)

##### `import A` 와 `from A import B`의 차이

```python
import json                      # json 상자를 통째로 가져온다
json.dumps(...)                  # → 쓸 때 상자 이름을 앞에 붙여야 한다

from datetime import datetime    # datetime 상자에서 datetime 하나만 꺼낸다
datetime.strptime(...)           # → 꺼낸 것을 바로 이름으로 쓴다
```

**`datetime`이 특히 헷갈리는 이유:** `datetime`이라는 *상자(모듈)* 안에 `datetime`이라는 *도구(클래스)*가 들어 있다. 이름이 같다.

```python
import datetime
datetime.datetime.strptime(...)      # 상자.도구.기능 → 길다

from datetime import datetime
datetime.strptime(...)               # 도구.기능 → 짧다 (이 방식을 쓴다)
```

##### 이 프로젝트에서 가져올 4가지

| 가져올 것 | 언제 쓰나 |
|---|---|
| `json` | Phase 4에서 데이터를 파일에 저장/불러올 때, Phase 6에서 도구 결과를 Gemini에 보낼 때 |
| `os` | `.env`에서 읽어온 값(API 키)을 꺼낼 때 |
| `datetime` (클래스) | 날짜 형식이 올바른지 검사할 때(`strptime`), 오늘 날짜를 구할 때(`now`) |
| `load_dotenv` (함수) | `.env` 파일을 읽어올 때. 파이썬 기본 창고가 아니라 **따로 설치한 `dotenv` 패키지**에 들어 있다 |

> 관례상 **파이썬 기본 라이브러리(`json`, `os`, `datetime`)를 먼저 쓰고, 한 줄 띄운 뒤 설치해서 쓰는 외부 패키지(`dotenv`)를 쓴다.** 문법으로 강제되진 않지만 실무 코드는 거의 다 이 순서다.

##### `.env`와 `load_dotenv`가 왜 필요한가

API 키를 코드에 직접 쓰면 이렇게 된다.

```python
api_key = "AIzaSyD-실제키가여기에..."      # ❌ 절대 하면 안 된다
```

이 코드를 GitHub에 올리는 순간 **키가 전 세계에 공개된다.** 남이 내 키로 API를 호출하면 요금은 나에게 청구된다.

그래서 키는 코드가 아니라 **`.env`라는 별도 파일**에 적어두고, 그 파일은 GitHub에 올리지 않는다(`.gitignore`에 등록). 코드에는 "그 파일에서 읽어와"라고만 쓴다.

**`load_dotenv()`가 실제로 하는 일:** `.env` 파일을 열어서 그 안의 `이름=값` 줄들을 읽고, **프로그램의 "환경변수" 목록에 올려둔다.**
여기까지가 전부다. 값을 돌려주지는 않는다. (값을 꺼내는 건 그다음 `os.getenv`의 일이다)

**괄호 안에 들어가는 것:** `.env` 파일의 **경로(문자열)**.

```python
load_dotenv()                    # 아무것도 안 주면 현재 폴더부터 위로 찾아 올라간다
load_dotenv("../.env")           # 경로를 직접 지정할 수도 있다
```

우리 `.env`는 이 프로젝트 폴더가 아니라 **상위 폴더(`260811`)**에 있으므로 경로를 적어주는 쪽이 확실하다.

> 윈도우 경로인데 `\` 대신 `/`를 쓰는 이유: 파이썬 문자열에서 `\`는 특수문자의 시작 기호다. `"C:\new"`라고 쓰면 `\n`이 줄바꿈으로 해석되어 경로가 깨진다. `/`를 쓰면 이런 문제가 없고 윈도우에서도 정상 동작한다.

##### 값 꺼내기 — `os.getenv`

`load_dotenv`가 올려둔 환경변수를 꺼내는 함수다.

```python
os.getenv("GEMINI_API_KEY")   # 있으면 그 값(문자열), 없으면 None
```

`.env`에 적은 **이름**을 문자열로 넣으면 값이 나온다.

**확인 방법:** 키 값을 그대로 `print`하면 화면에 키가 노출된다. 그래서 값 자체가 아니라 **"값이 있는지 없는지"**만 확인한다.

```python
print(os.getenv("GEMINI_API_KEY") is not None)   # True면 성공
```

> 지금은 Gemini를 쓰지 않지만 여기서 미리 확인해둔다. Phase 5에서 처음 확인하면, 에러가 났을 때 그게 키 문제인지 코드 문제인지 구분하기 어렵다.

---

##### ✍️ 아래 코드 셀에 작성할 것

1. 파이썬 기본 라이브러리 3개 가져오기 (`json`, `os`, `datetime`의 `datetime`)
2. 한 줄 띄우고 `dotenv`에서 `load_dotenv` 가져오기
3. `load_dotenv`로 상위 폴더의 `.env` 읽기
4. 키가 제대로 읽혔는지 `True`/`False`로 출력

</details>

In [2]:
# 1) 파이썬 기본 라이브러리 가져오기 (json / os / datetime의 datetime)
import json
import os
from datetime import datetime


# 2) 외부 패키지에서 load_dotenv 가져오기
from dotenv import load_dotenv 

# 3) 상위 폴더의 .env 읽어오기
load_dotenv()

# 4) GEMINI_API_KEY가 읽혔는지 True/False로 확인 출력
print(os.getenv("GEMINI_API_KEY")is not None)

True


##### 💡 1-1 준비 — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
import json
import os
from datetime import datetime

from dotenv import load_dotenv

load_dotenv()

print(os.getenv("GEMINI_API_KEY") is not None)
```

</details>

#### 1-2. `BudgetManager` 클래스와 `__init__`

<details>
<summary>설명 펼치기</summary>

##### 지금 무엇을 만드는가

이 프로젝트의 **본체**를 만든다. 위 로드맵 그림에서 맨 아래에 있던 *"실제로 보관하는 곳"*이 바로 이것이다.

앞으로 만들 도구 5개는 **전부 같은 데이터를 함께 쓴다.**

| 도구 | 무엇을 건드리나 |
|---|---|
| `add_transaction` | 거래 목록에 **추가**, ID 번호를 가져다 씀 |
| `search_transactions` | 거래 목록을 **읽음** |
| `update_transaction` | 거래 목록을 **고침** |
| `set_budget` | 예산을 **저장** |
| `get_budget_status` | 거래 목록과 예산을 **함께 읽음** |

이 다섯이 공유하는 데이터 — **거래 목록, 다음 ID, 카테고리 목록, 예산** — 를 한 곳에 모아두는 것이 `BudgetManager`다.

##### 왜 하필 지금인가

> **담을 그릇이 없으면, 담는 기능을 만들 수 없다.**

`add_transaction`을 먼저 만들려고 하면 첫 줄에서 막힌다. *"거래를 **어디에** 추가하지?"*
그 "어디"를 지금 만드는 것이다.

1-1은 사실 프로젝트라기보다 **재료 준비**였다. 창고에서 연장을 꺼내온 것뿐이다.
**실제 프로젝트의 첫 벽돌이 여기다.**

##### 이번에 만드는 범위는 "빈 그릇"까지

클래스를 지금 전부 만드는 게 아니다. 지금은 `__init__`만 — **속성 6개를 자리 잡아두는 것**까지다.

```
1-2   →  빈 그릇            (속성 6개)          ← 지금 여기
1-3   →  그릇을 지키는 규칙   (검증 함수 4개)
2~3   →  그릇을 다루는 기능   (도구 5개)
4     →  그릇을 파일에 저장   (_save / _load)
```

기능(메서드)은 1-3부터 **이 클래스 안에 하나씩 채워 넣는다.** 그래서 이 셀이 계속 자라난다.

##### 클래스를 왜 쓰나

함수만으로도 가계부를 만들 수는 있다. 하지만 이 프로그램에서는 **거래 목록, 다음 ID, 카테고리 목록, 예산** 네 종류의 데이터를 여러 함수가 함께 읽고 고쳐야 한다.

이걸 전부 바깥의 전역 변수로 두면 문제가 생긴다.
- 어느 함수가 언제 값을 바꿨는지 추적하기 어렵다.
- 테스트용 데이터와 실제 데이터를 따로 두기 어렵다.
- 함수마다 `global` 선언을 붙여야 해서 지저분해진다.

**클래스는 "데이터 + 그 데이터를 다루는 함수"를 한 덩어리로 묶는 도구다.** 묶어두면 `manager` 하나만 들고 다니면 그 안의 데이터와 기능이 전부 따라온다.

##### `__init__`이 하는 일

`BudgetManager()` 라고 **부르는 순간 자동으로 실행되는 함수**다. 인스턴스가 처음 가질 값들을 여기서 정한다.

이름 앞뒤의 밑줄 두 개(`__`)는 "파이썬이 특별하게 취급하는 이름"이라는 표시다. 우리가 직접 `manager.__init__()` 이라고 부를 일은 없다.

##### `self`가 뭔가

**지금 만들어지고 있는 인스턴스 자기 자신**을 가리키는 이름이다.

```python
self.transactions = []     # 이 인스턴스의 transactions 칸에 빈 리스트를 넣어라
transactions = []          # ❌ __init__이 끝나면 사라지는 임시 변수
```

`self.`를 붙여야 **다른 메서드에서도 꺼내 쓸 수 있는 값**이 된다. 안 붙이면 그 함수 안에서만 살다 사라진다. 이 프로젝트에서 가장 많이 하는 실수 중 하나다.

메서드를 정의할 때는 첫 번째 매개변수로 항상 `self`를 적지만, **부를 때는 넘기지 않는다.** 파이썬이 알아서 채운다.

```python
def _validate_date(self, date_str):   # 정의할 때는 self를 쓴다
    ...

manager._validate_date("2026-08-27")  # 부를 때는 date_str만 넘긴다
```

##### 기본값 매개변수

```python
def __init__(self, data_file="data.json"):
```

`=` 뒤에 값을 적어두면 **그 매개변수는 선택**이 된다.

```python
BudgetManager()                          # data_file은 "data.json"
BudgetManager("test_data.json")          # data_file은 "test_data.json"
```

굳이 매개변수로 뺀 이유는, 나중에 실제 데이터를 건드리지 않고 **테스트용 파일로 따로 실험**하기 위해서다.

##### ⚠️ 지금은 넣지 말 것

기획서 4-2절에는 `__init__` 마지막에 `self._load()`가 있지만, **아직 `_load` 함수가 없어서 지금 넣으면 에러가 난다.** Phase 4에서 만든 뒤에 추가한다.

---

##### ⭐ 이 셀은 앞으로 계속 자라난다

노트북에서 클래스는 **셀을 실행할 때 통째로 다시 정의된다.** 메서드를 여러 셀에 나눠 쓸 수 없다.

그래서 앞으로 만들 모든 메서드(`add_transaction`, `search_transactions` …)는 **전부 아래 이 셀 안에 추가**하고, 추가할 때마다 **이 셀을 다시 실행**한다. 다시 실행하지 않으면 새로 쓴 메서드가 반영되지 않는다.

---

##### 담아야 할 6가지

| 속성 | 처음 값 | 역할 |
|---|---|---|
| `self.data_file` | 매개변수로 받은 값 | 저장할 파일 이름 |
| `self.transactions` | 빈 리스트 `[]` | 거래 딕셔너리들이 쌓일 곳 |
| `self.next_id` | `1` | 다음 거래에 붙일 번호 |
| `self.expense_categories` | 지출 카테고리 리스트 | 식비 / 교통비 / 주거비 / 문화생활 / 기타 |
| `self.income_categories` | 수입 카테고리 리스트 | 월급 / 용돈 / 부수입 / 기타 |
| `self.budgets` | 빈 딕셔너리 `{}` | `{"2026-08": {"식비": 300000}}` 형태로 쌓임 |

> **카테고리를 파일 맨 위의 상수로 두지 않고 `self.`에 넣는 이유:** 나중에 "카테고리 추가" 기능을 붙일 때, 인스턴스 속성이면 리스트에 값을 추가하고 저장만 하면 된다. 상수로 두면 실행 중에 바꿔도 JSON에 저장되지 않아 재실행하면 사라진다.

##### ✍️ 아래 코드 셀에 작성할 것

`BudgetManager` 클래스를 만들고, `__init__` 안에서 위 표의 6가지 속성을 `self.`에 담는다.

</details>

In [3]:
class BudgetManager:

    def __init__(self, data_file="data.json"):
    
        # 1) 저장할 파일 이름
        self.data_file = data_file

        # 2) 거래 목록 (빈 리스트)
        self.transactions = []

        # 3) 다음 거래 ID (1부터 시작)
        self.next_id = 1

        # 4) 지출 카테고리 목록
        self.expense_categories = ['식비','교통비','주거비','문화생활','기타']

        # 5) 수입 카테고리 목록
        self.income_categories = ['월급','용돈','부수입','기타']

        # 6) 예산 (빈 딕셔너리)
        self.budgets = {}

        # self._load()  ← Phase 4에서 주석을 푼다



    # ------------------------------------------------------------------
    # 검증 헬퍼 (1-3에서 여기에 추가)
    # ------------------------------------------------------------------


    # ------------------------------------------------------------------
    # 도구 함수 (Phase 2~3에서 여기에 추가)
    # ------------------------------------------------------------------


    # ------------------------------------------------------------------
    # 저장 / 불러오기 (Phase 4에서 여기에 추가)
    # ------------------------------------------------------------------

manager = BudgetManager()

##### 💡 1-2 클래스 뼈대 — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
class BudgetManager:

    def __init__(self, data_file="data.json"):
        self.data_file = data_file
        self.transactions = []
        self.next_id = 1
        self.expense_categories = ["식비", "교통비", "주거비", "문화생활", "기타"]
        self.income_categories = ["월급", "용돈", "부수입", "기타"]
        self.budgets = {}

        # self._load()   ← Phase 4에서 주석을 푼다


manager = BudgetManager()
```

</details>

##### ⚙️ 이 노트북을 다루는 규칙 — 딱 하나

<details>
<summary>설명 펼치기</summary>

위 클래스 셀 **맨 아래(들여쓰기 없이)**에 이 한 줄을 넣어 둔다.

```python
manager = BudgetManager()
```

그러면 규칙은 이것 하나로 끝난다.

> **클래스 셀을 고쳤으면, 그 셀을 다시 실행한다 (Shift + Enter).**

셀을 다시 실행하면 클래스가 새로 정의되고, 바로 아래 줄에서 `manager`도 자동으로 새로 만들어진다.
앞으로 "인스턴스를 다시 만들어라"는 말은 나오지 않는다.

**어느 셀을 다시 돌려야 할지 헷갈리면** 노트북 위쪽의 **▷▷ Run All(모두 실행)**을 누르면 된다.
위에서부터 전부 순서대로 다시 실행되므로 아무것도 꼬이지 않는다.

> Phase 5부터는 Gemini에 실제 요청이 나가서 비용이 발생하므로, 그때부터는 Run All 대신 필요한 셀만 실행한다.

##### 인스턴스를 만들어 확인

클래스는 **설계도**일 뿐이라, 실제로 쓰려면 그 설계도로 **인스턴스(실물)**를 만들어야 한다.

```python
manager = BudgetManager()
```

이렇게 하면 `__init__`이 실행되면서 속성들이 채워진 객체 하나가 만들어지고, 그걸 `manager`라는 이름으로 부르게 된다. 앞으로 모든 기능은 `manager.무엇()` 형태로 쓴다.

##### ✍️ 아래 코드 셀에 작성할 것

인스턴스를 만들고, 속성 몇 개를 출력해서 의도한 값이 들어갔는지 눈으로 확인한다.

</details>

In [4]:
# 클래스 셀에서 만들어진 manager의 속성들이 의도대로 채워졌는지 눈으로 확인한다.
# manager.속성이름 형태로 꺼내서 출력하면 된다.

print("파일 이름   :", manager.data_file)
print("거래 목록   :", manager.transactions)
print("다음 ID     :", manager.next_id)
print("지출 카테고리:", manager.expense_categories)
print("수입 카테고리:", manager.income_categories)
print("예산        :", manager.budgets)

파일 이름   : data.json
거래 목록   : []
다음 ID     : 1
지출 카테고리: ['식비', '교통비', '주거비', '문화생활', '기타']
수입 카테고리: ['월급', '용돈', '부수입', '기타']
예산        : {}


#### 1-3. 검증 함수 4개

<details open>
<summary>설명 펼치기</summary>

`add_transaction`, `update_transaction`, `set_budget` 세 도구가 **똑같은 검사를 반복**한다.
"금액이 올바른가", "날짜 형식이 맞는가" 같은 것들이다.

같은 코드를 세 군데에 복사해두면 규칙이 바뀔 때 세 군데를 다 고쳐야 하고, 한 군데를 빠뜨리면 버그가 된다.
**한 곳에 모아두면 고칠 때 한 곳만 고치면 된다.**

| 만들 함수 | 무엇을 검사 | 돌려주는 값 |
|---|---|---|
| `_validate_amount` | 금액 | 정수, 실패하면 `None` |
| `_validate_date` | 날짜 (`2026-08-27`) | `True` / `False` |
| `_validate_month` | 월 (`2026-08`) | `True` / `False` |
| `_validate_category` | 카테고리 | `True` / `False` |

이름 앞의 `_`는 **"내부용이니 바깥에서 직접 쓰지 마세요"**라는 개발자들 사이의 약속이다.
파이썬이 막아주는 건 아니고 표시일 뿐이다. Gemini에 등록할 도구와 눈으로 구분하려고 쓴다.

##### 이번엔 클래스 셀로 올라가지 않는다

노트북에서 클래스 안에 메서드를 하나씩 추가하려면 **클래스 전체를 매번 다시 써야** 한다. 불편하다.

그래서 앞으로는 이렇게 한다.

```python
def _validate_amount(self, amount):     # ① 함수를 평소처럼 만들고
    ...

BudgetManager._validate_amount = _validate_amount   # ② 마지막 줄에서 클래스에 붙인다
```

②번 줄은 **"이 함수를 `BudgetManager`의 메서드로 등록한다"**는 뜻이다.
등록하고 나면 이미 만들어 둔 `manager`에서도 바로 쓸 수 있다.

```python
manager._validate_amount(12000)
```

첫 번째 매개변수를 `self`로 쓰는 건 클래스 안에 넣을 때와 똑같다. 붙이고 나면 진짜 메서드가 되기 때문이다.

> Phase 9에서 `.py`로 옮길 때는 이 함수들을 클래스 안으로 모아 넣는다. 지금은 셀을 나눠 쓰기 위한 방법이다.

> ⚠️ 클래스 셀을 다시 실행하면 클래스가 새로 만들어져서 **붙여둔 메서드가 사라진다.**
> 그럴 땐 아래 셀들도 다시 실행하면 된다. (헷갈리면 Run All)

##### 공통 문법 — `try` / `except`

파이썬은 에러가 나면 **그 자리에서 프로그램이 멈춘다.**

```python
int("사과")      # ValueError 발생 → 여기서 중단
print("다음")     # 실행조차 안 됨
```

`try` / `except`는 에러가 날 수 있는 코드를 감싸서, 에러가 나면 죽는 대신 **다른 길로 가게** 한다.

```python
try:
    number = int("사과")     # 여기서 에러가 나면
except ValueError:
    number = 0               # 즉시 이쪽으로 넘어온다
```

`except` 뒤에는 **잡을 에러 종류**를 적는다. 여러 개면 괄호로 묶는다.

```python
except (TypeError, ValueError):
```

- `ValueError` — 타입은 맞는데 **값이 이상할 때** (`int("사과")`)
- `TypeError` — **타입 자체가 안 맞을 때** (`int(None)`)

Gemini가 무엇을 보낼지 모르므로 검증 함수는 **둘 다 잡는다.**

##### ① `_validate_amount` — 금액 검사

Gemini는 금액을 `12000`, `"12000"`, `12000.0` 중 **어느 형태로든** 보낼 수 있다.
셋 다 받아서 정수로 바꿔주고, 이상하면 거절한다.

**왜 `int(float(...))` 인가**

```python
int("12000")        # 12000  → 된다
int("12000.5")      # ❌ ValueError — int()는 소수점 있는 문자열을 못 받는다
float("12000.5")    # 12000.5 → 된다
int(12000.5)        # 12000  → float를 int로는 자를 수 있다
```

그래서 **`float`을 한 번 거친 뒤 `int`로 자르면** 세 형태를 모두 안전하게 처리할 수 있다.

**왜 `True`/`False`가 아니라 값을 돌려주나**

이 함수만 검사뿐 아니라 **변환까지** 해준다. 부르는 쪽에서 이렇게 쓸 수 있다.

```python
value = self._validate_amount(amount)
if value is None:
    return {...오류...}

# 여기서부터 value는 안전한 정수다

```

`True`/`False`만 주면 부르는 쪽에서 변환을 또 해야 한다.

**거절할 경우:** 변환 실패, 그리고 `0` 이하. (0원이나 마이너스 지출은 의미가 없다)

</details>

In [5]:
def _validate_amount(self, amount):
    # 1) amount를 정수로 바꿔 본다. 금액이 정수형으로 잘 들어왔는지 확인하는 코드.
    #    - try: 안에서 float으로 바꾸고 다시 int로 바꿔서 변수(예: value)에 담는다
    #    - except (TypeError, ValueError): 안에서는 return None
    try:
        value = int(float(amount))
    except (TypeError, ValueError):
        return None
    
    # 2) value가 0 이하이면 return None
    if value <= 0:
        return None
    
    # 3) 여기까지 왔으면 올바른 금액이다 → value를 return
    return value

BudgetManager._validate_amount = _validate_amount   # 클래스에 등록


# ---- 확인 ----
print(manager._validate_amount(12000))      # 12000
print(manager._validate_amount("12000"))    # 12000
print(manager._validate_amount(12000.0))    # 12000
print(manager._validate_amount("만이천원"))  # None
print(manager._validate_amount(-500))       # None
print(manager._validate_amount(0))          # None

12000
12000
12000
None
None
None


##### 💡 _validate_amount — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def _validate_amount(self, amount):
    # 1) 정수로 변환 시도
    try:
        value = int(float(amount))
    except (TypeError, ValueError):
        return None

    # 2) 0 이하는 거절
    if value <= 0:
        return None

    # 3) 올바른 금액
    return value


BudgetManager._validate_amount = _validate_amount
```

</details>

##### ② `_validate_date` — 날짜 검사

<details open>
<summary>설명 펼치기</summary>

`strptime`은 **str-p-time**, "문자열(str)을 해석(parse)해서 시간(time)으로 만든다"는 뜻이다.

```python
datetime.strptime("2026-08-27", "%Y-%m-%d")
```

두 번째 인자가 **형식 설명서**다.

| 기호 | 의미 |
|---|---|
| `%Y` | 네 자리 연도 (2026) |
| `%m` | 두 자리 월 (08) |
| `%d` | 두 자리 일 (27) |

형식이 다르거나 **실제로 없는 날짜**면 `ValueError`를 던진다.

```python
datetime.strptime("2026-13-01", "%Y-%m-%d")   # 13월은 없다 → ValueError
datetime.strptime("2026-02-30", "%Y-%m-%d")   # 2월 30일은 없다 → ValueError
```

자릿수를 직접 세는 것보다 정확하고, **윤년까지 알아서 처리**해준다.

우리는 변환 결과가 필요한 게 아니라 "올바른 날짜인가"만 알면 되므로,
**성공하면 `True`, 에러가 잡히면 `False`**를 돌려준다. 변환 결과는 버린다.

</details>

In [6]:
def _validate_date(self, date_str):
    # 1) try: 안에서 datetime.strptime(date_str, "%Y-%m-%d") 를 실행하고
    #    바로 return True   (변환 결과는 쓰지 않는다)

    try:
        datetime.strptime(date_str, "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False



    # 2) except (ValueError, TypeError): 안에서 return False



BudgetManager._validate_date = _validate_date


# ---- 확인 ----
print(manager._validate_date("2026-08-27"))   # True
print(manager._validate_date("2026-13-01"))   # False  (13월은 없다)
print(manager._validate_date("2026-02-30"))   # False  (2월 30일은 없다)
print(manager._validate_date("8월 27일"))      # False  (형식이 다르다)
print(manager._validate_date(None))           # False  (TypeError를 잡는다)

True
False
False
False
False


##### 💡 _validate_date — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def _validate_date(self, date_str):
    try:
        datetime.strptime(date_str, "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False


BudgetManager._validate_date = _validate_date
```

</details>

##### ③ `_validate_month` — 월 검사

<details open>
<summary>설명 펼치기</summary>

②와 **똑같은 구조**이고 형식 문자열만 `"%Y-%m"`으로 바뀐다.

예산은 `"2026-08"`처럼 **월 단위**로 관리하기 때문에 따로 필요하다.

```python
datetime.strptime("2026-08", "%Y-%m")     # OK
datetime.strptime("2026-13", "%Y-%m")     # ❌ 13월은 없다 (에러)
```

</details>

In [7]:
# def _validate_month(self, month_str):
    # ②와 같은 모양. 형식 문자열만 "%Y-%m"
def _validate_month(self, month_str):
    try:
        datetime.strptime(month_str, "%Y-%m")
        return True
    except (TypeError, ValueError):
        return False    



BudgetManager._validate_month = _validate_month


# ---- 확인 ----
print(manager._validate_month("2026-08"))      # True
print(manager._validate_month("2026-8"))       # True   (한 자리 월도 통과된다)
print(manager._validate_month("2026-13"))      # False  (13월은 없다)
print(manager._validate_month("2026-08-27"))   # False  (일자까지 있으면 형식이 다르다)

True
True
False
False


##### 💡 _validate_month — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def _validate_month(self, month_str):
    try:
        datetime.strptime(month_str, "%Y-%m")
        return True
    except (ValueError, TypeError):
        return False


BudgetManager._validate_month = _validate_month
```

</details>

##### ④ `_validate_category` — 카테고리 검사

<details open>
<summary>설명 펼치기</summary>

카테고리는 **거래 유형에 따라 허용 목록이 다르다.**
지출에 "월급"을 쓰면 안 되고, 수입에 "식비"를 쓰면 안 된다.

리스트에 들어있는지는 `in`으로 검사한다.

```python
"식비" in ["식비", "교통비"]     # True
"월급" in ["식비", "교통비"]     # False
```

**세 갈래로 나뉜다.**

| `transaction_type` | 무엇을 볼까 |
|---|---|
| `"지출"` | `self.expense_categories` 안에 있는가 |
| `"수입"` | `self.income_categories` 안에 있는가 |
| 그 외 | `False` (유형 자체가 잘못됐으니 카테고리도 유효할 수 없다) |

> 세 번째 갈래를 빠뜨리면 이상한 유형이 왔을 때 함수가 아무것도 돌려주지 않고 끝나서 `None`이 나온다. 반드시 넣는다.

`if` 안에서 바로 `return`하면 `else` 없이도 깔끔하게 쓸 수 있다.

```python
if transaction_type == "지출":
    return category in self.expense_categories
```

`in` 비교의 결과가 이미 `True`/`False`이므로 **그대로 `return`하면 된다.**

</details>

In [8]:
def _validate_category(self, category, transaction_type):
    # 카테고리 애러검사.
    # 1) transaction_type이 "지출"이면
    #    → category가 self.expense_categories 안에 있는지를 return
    if transaction_type == "지출":
        return category in self.expense_categories

    # 2) transaction_type이 "수입"이면
    #    → category가 self.income_categories 안에 있는지를 return
    elif transaction_type == "수입":
        return category in self.income_categories
    # 3) 둘 다 아니면 return False
    else:
        return False



BudgetManager._validate_category = _validate_category


# ---- 확인 ----
print(manager._validate_category("식비", "지출"))     # True
print(manager._validate_category("월급", "지출"))     # False  (수입 카테고리다)
print(manager._validate_category("월급", "수입"))     # True
print(manager._validate_category("치킨", "지출"))     # False  (목록에 없다)
print(manager._validate_category("식비", "이상한값"))  # False  (유형이 잘못됐다)

True
False
True
False
False


##### 💡 _validate_category — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def _validate_category(self, category, transaction_type):
    if transaction_type == "지출":
        return category in self.expense_categories

    if transaction_type == "수입":
        return category in self.income_categories

    return False


BudgetManager._validate_category = _validate_category
```

</details>

##### ✅ Phase 1 완료 확인

<details open>
<summary>설명 펼치기</summary>

위 네 셀의 출력이 오른쪽 주석과 전부 일치하면 Phase 1은 끝이다.

하나라도 다르면 **그 함수만** 고치면 된다. 다른 함수에 영향을 주지 않는다.

> 자주 나오는 에러
> - `AttributeError: 'BudgetManager' object has no attribute 'income_categories'`
>   → 클래스 셀의 속성 이름에 오타가 있다
> - `NameError: name 'manager' is not defined`
>   → 클래스 셀을 아직 실행하지 않았다
> - `NameError: name 'datetime' is not defined`
>   → 1-1 셀을 실행하지 않았거나 `from datetime import datetime`이 아니다

</details>

### Phase 2 — 핵심 도구 3개 (등록 / 검색 / 수정)

<details open>
<summary>설명 펼치기</summary>

여기서부터 만드는 함수들이 **나중에 Gemini가 호출할 도구**다.
아직 Gemini는 등장하지 않는다. 우리가 직접 불러서 확인한다.

##### 먼저 정할 것 — 반환 형식 규약

도구가 저마다 다른 모양으로 결과를 돌려주면, Agent가 성공/실패를 판단하는 방법도 제각각이 된다.
그래서 **모든 도구가 같은 모양으로 돌려주도록** 규칙을 정한다.

```python
# 성공
{"ok": True,  "data": {...},              "message": "식비 12,000원을 등록했습니다."}

# 실패
{"ok": False, "error": "INVALID_AMOUNT",  "message": "금액은 0보다 큰 숫자여야 합니다."}
```

| 키 | 역할 |
|---|---|
| `ok` | 성공/실패를 한 줄로 판단하는 기준 |
| `data` | 실제 결과물. Gemini가 이걸 읽고 답변을 만든다 |
| `error` | 오류 종류 코드 (디버깅·로그용) |
| `message` | 사람이 읽는 문장. Gemini가 거의 그대로 전달한다 |

**오류 코드는 이 아홉 가지를 쓴다.**

`INVALID_TYPE` · `INVALID_CATEGORY` · `INVALID_AMOUNT` · `INVALID_DATE` · `INVALID_MONTH`
`NOT_FOUND` · `NOTHING_TO_UPDATE` · `NO_BUDGET_SET` · `NO_DATA`

##### `message`는 정보를 많이 담을수록 좋다

```python
"등록했습니다."                                    # 부실
"[1] 2026-08-27 식비 지출 12,000원을 등록했습니다."   # 좋다
```

특히 **등록할 때 ID를 넣어주면** 이후 "1번 거래 수정해줘" 같은 대화가 가능해진다.

> 숫자에 쉼표를 넣으려면 `f"{amount:,}"` 를 쓴다. `f"{12000:,}"` → `12,000`

</details>

#### 2-1. `add_transaction` — 거래 등록

<details open>
<summary>설명 펼치기</summary>

받은 값을 **검사하고, 번호를 붙여서, 목록에 넣는다.**

```python
def add_transaction(self, transaction_type, category, amount, date, description=""):
```

`description`에만 기본값을 준 이유: 기본값이 있으면 Gemini가 그 정보를 못 찾았을 때 **생략하고 호출할 수 있다.**
금액·날짜·카테고리는 없으면 거래가 성립하지 않으므로 기본값을 주지 않는다. 그래야 Gemini가 사용자에게 되묻는다.

##### 순서

```
1. transaction_type이 "수입"/"지출" 중 하나인가?   → 아니면 INVALID_TYPE
2. amount를 정수로 바꿀 수 있는가?                → 아니면 INVALID_AMOUNT
3. date 형식이 올바른가?                         → 아니면 INVALID_DATE
4. category가 그 유형의 목록에 있는가?            → 아니면 INVALID_CATEGORY
5. 거래 딕셔너리를 만들어 목록에 추가
6. next_id를 1 증가
7. 성공 반환
```

**검사를 전부 끝내고 나서 저장한다.** 저장부터 하면 잘못된 데이터가 이미 들어간 뒤가 된다.
`transaction_type`을 가장 먼저 검사하는 이유는, 카테고리 검사가 유형을 필요로 하기 때문이다.

##### 주의 3가지

- 매개변수는 `transaction_type`이지만 **딕셔너리 키는 `"type"`**이다
- `amount`는 **`_validate_amount`가 돌려준 값**을 넣는다 (원래 받은 값을 넣으면 `"12000"`이 그대로 저장된다)
- `self.next_id += 1`을 빠뜨리면 모든 거래 ID가 1이 되어 수정·삭제가 전부 망가진다

##### 오류 메시지에 도움말을 담는다

```python
f"'{category}'는 지출 카테고리가 아닙니다. 사용 가능: {', '.join(self.expense_categories)}"
```

쓸 수 있는 값을 알려주면 Gemini가 스스로 고쳐서 다시 호출할 수 있다.
`", ".join(리스트)` 는 리스트를 `식비, 교통비, 주거비` 형태의 문자열로 만든다.

</details>

In [9]:
def add_transaction(self, transaction_type, category, amount, date, description=""):

    # 1) 거래 유형이 "수입" / "지출" 중 하나인지 확인한다
    #    왜  : 뒤에 나오는 카테고리 검사가 이 값을 보고 "지출 목록을 볼지 수입 목록을 볼지"를 정한다.
    #          유형이 엉뚱하면 그 뒤 검사가 전부 의미 없어지므로 가장 먼저 막는다.
    #    방법: transaction_type not in ("수입", "지출") 이면
    #          {"ok": False, "error": "INVALID_TYPE", "message": ...} 를 return 한다.
    #          return 하는 순간 함수가 끝나므로 아래 코드는 실행되지 않는다.
# 1) 거래 유형 검사
    if transaction_type not in ("수입", "지출"):
        return {"ok": False, "error": "INVALID_TYPE",
                "message": "거래 유형은 '수입' 또는 '지출'이어야 합니다."}



    # 2) 금액 검사·변환 함수를 불러온다 (변환은 그 함수 안에서 일어난다)
    #    왜  : Gemini는 12000 / "12000" / 12000.0 중 아무 형태로나 보낼 수 있다.
    #          저장할 때 정수로 통일해두지 않으면 나중에 합계를 구할 때 문자열끼리 더해져 깨진다.
    #    방법: value = self._validate_amount(amount) 로 받는다.
    #          None이면 INVALID_AMOUNT 를 return.
    #          ⚠️ 여기서 얻은 value 를 5)에서 저장에 쓴다. 원래 amount 를 쓰면 "12000"이 그대로 들어간다.

    # 2) 금액 검사 + 변환
    value = self._validate_amount(amount)
    if value is None:
        return {"ok": False, "error": "INVALID_AMOUNT",
                "message": "금액은 0보다 큰 숫자여야 합니다."}



    # 3) 날짜 형식을 검사한다
    #    왜  : 날짜가 "2026-08-27" 형태로 통일돼야 검색(문자열 비교)과 월별 집계(startswith)가 동작한다.
    #          하나라도 다른 모양이 섞이면 그 거래는 어떤 검색에도 안 잡힌다.
    #    방법: self._validate_date(date) 가 False면 INVALID_DATE 를 return.
    if not self._validate_date(date):
        return {"ok": False, "error": "INVALID_DATE",
                "message": "날짜는 YYYY-MM-DD 형식이어야 합니다. 예: 2026-08-27"}



    # 4) 카테고리가 그 유형의 목록에 있는지 검사한다
    #    왜  : 목록에 없는 카테고리를 허용하면, 그 지출이 어느 예산에도 잡히지 않아
    #          "예산은 남았는데 통장은 빈" 상태가 된다.
    #    방법: self._validate_category(category, transaction_type) 가 False면 INVALID_CATEGORY 를 return.
    #          message에 쓸 수 있는 목록을 ", ".join(...) 으로 붙여주면
    #          Gemini가 스스로 고쳐서 다시 호출할 수 있다.
    
    # 4) 카테고리 검사
    if not self._validate_category(category, transaction_type):
        if transaction_type == "지출":
            allowed = ", ".join(self.expense_categories)
        else:
            allowed = ", ".join(self.income_categories)
        return {"ok": False, "error": "INVALID_CATEGORY",
                "message": f"'{category}'는 {transaction_type} 카테고리가 아닙니다. 사용 가능: {allowed}"}


    # 5) 거래 딕셔너리를 만든다
    #    왜  : 여기까지 왔다는 건 값이 전부 안전하다는 뜻이다. 이제 실제로 저장할 데이터를 만든다.
    #          이 프로젝트에서 "거래 한 건" = "딕셔너리 하나" 로 표현하기로 정했다(기획서 4-1).
    #          검색·수정·예산 집계가 전부 이 모양을 전제로 동작하므로 키 이름을 정확히 맞춰야 한다.
    #    방법: 키 6개를 채운 딕셔너리를 변수(예: transaction)에 담는다.
    #          "id"          → self.next_id        (아직 아무도 안 쓴 번호)
    #          "type"        → transaction_type    (매개변수 이름과 키 이름이 다르다)
    #          "category"    → category
    #          "amount"      → 2)에서 얻은 value
    #          "description" → description
    #          "date"        → date
    
    # 5) 거래 만들기
    transaction = {
        "id": self.next_id,
        "type": transaction_type,
        "category": category,
        "amount": value,
        "description": description,
        "date": date,
    }



    # 6) 목록에 추가하고 다음 번호로 넘긴다
    #    왜  : 딕셔너리를 만들기만 하면 함수가 끝날 때 사라진다. self.transactions 에 넣어야 남는다.
    #          next_id 를 올리지 않으면 다음 거래도 같은 번호를 받아,
    #          "3번 수정해줘"가 엉뚱한 거래를 고치게 된다.
    #    방법: self.transactions.append(...) 한 뒤 self.next_id += 1
    self.transactions.append(transaction)
    self.next_id += 1

    # 7) 성공 반환
    return {
        "ok": True,
        "data": transaction,
        "message": f"[{transaction['id']}] {date} {category} {transaction_type} {value:,}원을 등록했습니다.",
    }


BudgetManager.add_transaction = add_transaction



    # 7) 성공 결과를 규약대로 돌려준다
    #    왜  : 모든 도구가 같은 모양으로 돌려줘야 Agent loop가 성공/실패를 한 가지 방법으로 판단한다.
    #          message에 ID를 넣어두면 Gemini가 사용자에게 번호를 알려줄 수 있고,
    #          그래야 이후 "1번 거래 수정해줘" 같은 대화가 가능해진다.
    #    방법: {"ok": True, "data": 만든 거래, "message": f"[{...}] ... 등록했습니다."}
    #          금액에 천 단위 쉼표를 넣으려면 f"{value:,}" 를 쓴다.



BudgetManager.add_transaction = add_transaction


# ---- 확인: 나중 테스트에 쓸 데이터를 만들어 둔다 ----
print(manager.add_transaction("지출", "식비", 12000, "2026-08-27", "점심")["message"])
print(manager.add_transaction("지출", "식비", 5000, "2026-08-27", "카페 아메리카노")["message"])
print(manager.add_transaction("지출", "교통비", 1500, "2026-08-28", "지하철")["message"])
print(manager.add_transaction("수입", "월급", 3000000, "2026-08-25", "8월 급여")["message"])

print()
for t in manager.transactions:
    print(t)

print()
# ---- 오류 상황 ----
print(manager.add_transaction("소비", "식비", 1000, "2026-08-27"))          # INVALID_TYPE
print(manager.add_transaction("지출", "식비", "만원", "2026-08-27"))         # INVALID_AMOUNT
print(manager.add_transaction("지출", "식비", 1000, "2026-13-01"))          # INVALID_DATE
print(manager.add_transaction("지출", "치킨", 1000, "2026-08-27"))          # INVALID_CATEGORY

[1] 2026-08-27 식비 지출 12,000원을 등록했습니다.
[2] 2026-08-27 식비 지출 5,000원을 등록했습니다.
[3] 2026-08-28 교통비 지출 1,500원을 등록했습니다.
[4] 2026-08-25 월급 수입 3,000,000원을 등록했습니다.

{'id': 1, 'type': '지출', 'category': '식비', 'amount': 12000, 'description': '점심', 'date': '2026-08-27'}
{'id': 2, 'type': '지출', 'category': '식비', 'amount': 5000, 'description': '카페 아메리카노', 'date': '2026-08-27'}
{'id': 3, 'type': '지출', 'category': '교통비', 'amount': 1500, 'description': '지하철', 'date': '2026-08-28'}
{'id': 4, 'type': '수입', 'category': '월급', 'amount': 3000000, 'description': '8월 급여', 'date': '2026-08-25'}

{'ok': False, 'error': 'INVALID_TYPE', 'message': "거래 유형은 '수입' 또는 '지출'이어야 합니다."}
{'ok': False, 'error': 'INVALID_AMOUNT', 'message': '금액은 0보다 큰 숫자여야 합니다.'}
{'ok': False, 'error': 'INVALID_DATE', 'message': '날짜는 YYYY-MM-DD 형식이어야 합니다. 예: 2026-08-27'}
{'ok': False, 'error': 'INVALID_CATEGORY', 'message': "'치킨'는 지출 카테고리가 아닙니다. 사용 가능: 식비, 교통비, 주거비, 문화생활, 기타"}


##### 💡 add_transaction — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def add_transaction(self, transaction_type, category, amount, date, description=""):
    # 1) 거래 유형 검사
    if transaction_type not in ("수입", "지출"):
        return {"ok": False, "error": "INVALID_TYPE",
                "message": "거래 유형은 '수입' 또는 '지출'이어야 합니다."}

    # 2) 금액 검사 + 변환
    value = self._validate_amount(amount)
    if value is None:
        return {"ok": False, "error": "INVALID_AMOUNT",
                "message": "금액은 0보다 큰 숫자여야 합니다."}

    # 3) 날짜 검사
    if not self._validate_date(date):
        return {"ok": False, "error": "INVALID_DATE",
                "message": "날짜는 YYYY-MM-DD 형식이어야 합니다. 예: 2026-08-27"}

    # 4) 카테고리 검사
    if not self._validate_category(category, transaction_type):
        if transaction_type == "지출":
            allowed = ", ".join(self.expense_categories)
        else:
            allowed = ", ".join(self.income_categories)
        return {"ok": False, "error": "INVALID_CATEGORY",
                "message": f"'{category}'는 {transaction_type} 카테고리가 아닙니다. 사용 가능: {allowed}"}

    # 5) 거래 만들기
    transaction = {
        "id": self.next_id,
        "type": transaction_type,
        "category": category,
        "amount": value,
        "description": description,
        "date": date,
    }

    # 6) 목록에 추가하고 다음 번호로
    self.transactions.append(transaction)
    self.next_id += 1

    # 7) 성공 반환
    return {
        "ok": True,
        "data": transaction,
        "message": f"[{transaction['id']}] {date} {category} {transaction_type} {value:,}원을 등록했습니다.",
    }


BudgetManager.add_transaction = add_transaction
```

</details>

#### 2-2. `search_transactions` — 거래 검색

<details open>
<summary>설명 펼치기</summary>

명세서의 핵심 시연이 *"검색 후 수정"*이다. 사용자는 거래 ID를 모르므로,
**먼저 검색해서 ID를 알아낸 다음에야 수정할 수 있다.** 이 함수가 그 첫 단계다.

```python
def search_transactions(self, date=None, start_date=None, end_date=None,
                        category=None, keyword=None):
```

**매개변수 5개가 전부 기본값 `None`이다.** 사용자는 "8월에 식비 쓴 거"처럼 일부 조건만 말하기 때문이다.
`None`은 "이 조건은 안 쓴다"는 표시가 된다. 빈 문자열 `""`을 쓰면 "빈 값으로 검색"인지 "조건 없음"인지 구분이 안 된다.

##### 조건 걸러내기

전체 거래를 하나씩 보면서, **조건에 맞지 않으면 `continue`로 건너뛰고**, 끝까지 살아남은 것만 담는다.

```python
for t in self.transactions:
    if category is not None and t["category"] != category:
        continue        # 카테고리 조건이 있는데 다르다 → 이 거래는 제외
    results.append(t)
```

##### 날짜는 문자열끼리 비교하면 된다

```python
"2026-08-27" >= "2026-08-01"    # True
```

`YYYY-MM-DD`는 자릿수가 고정이라 **사전순이 곧 날짜순**이다. `datetime`으로 바꿀 필요가 없다.

`date`와 기간이 같이 오면 **`date`를 우선**한다. 특정 날짜를 콕 집은 게 더 구체적인 요청이기 때문이다.

##### 키워드는 부분 일치

```python
"카페" in "카페 아메리카노"     # True
```

양쪽을 `.lower()`로 낮춰서 비교하면 영어 메모의 대소문자도 무시할 수 있다.

##### 정렬

```python
results.sort(key=lambda t: (t["date"], t["id"]))
```

`key`에 튜플을 주면 **앞의 값으로 먼저, 같으면 뒤의 값으로** 정렬한다.
`lambda t: ...` 는 "이름 없는 짧은 함수"로, 각 항목에서 정렬 기준을 꺼내는 역할이다.

##### 0건이어도 실패가 아니다

**검색은 성공했고 결과가 없을 뿐이다.** `ok: True`, `count: 0`으로 반환한다.
여기서 `ok: False`를 주면 Gemini가 "오류가 났다"고 오해한다.

`count`를 따로 담는 이유는, Gemini가 **"여러 건이니 되물어야겠다"**를 즉시 판단하게 하기 위해서다.

</details>

In [10]:
def search_transactions(self, date=None, start_date=None, end_date=None,
                        category=None, keyword=None):

    # 1) 결과를 담을 빈 리스트를 만든다
    #    왜  : 원본 self.transactions 에서 직접 지우면 데이터가 사라진다.
    #          검색은 "고르는 일"이지 "지우는 일"이 아니므로, 조건에 맞는 것만 새 리스트에 모은다.
    #    방법: results = []
    results =[]


    # 2) 전체 거래를 하나씩 보면서, 조건에 맞지 않으면 건너뛴다
    #    왜  : 조건이 5개인데 사용자는 그중 일부만 말한다("8월에 식비 쓴 거").
    #          그래서 각 조건마다 "이 조건이 주어졌는가?"를 먼저 확인하고, 주어진 것만 적용해야 한다.
    #          맞는 것을 찾는 것보다 "틀린 것을 걸러내는" 방식이, 조건이 늘어나도 코드가 단순하다.
    #    방법: for t in self.transactions: 안에서 조건마다 이렇게 쓴다.
    #          - date 가 주어졌으면(is not None): t["date"] 와 다르면 continue
    #          - date 가 없을 때만 start_date / end_date 를 본다
    #            (기획서 규칙: 특정 날짜가 더 구체적인 요청이므로 date 를 우선한다)
    #            날짜는 "2026-08-27" 형태라 문자열 비교로 순서 판정이 된다
    #            → t["date"] < start_date 이면 너무 이르다, > end_date 이면 너무 늦다
    #          - category 가 주어졌으면: 다르면 continue
    #          - keyword 가 주어졌으면: t["description"] 안에 없으면 continue
    #            (양쪽 .lower() 로 낮춰서 비교하면 영어 대소문자를 무시할 수 있다)
    #          - 여기까지 살아남았으면 results 에 append
    for t in self.transactions:
        if date is not None: #잼미니가준 date 안에 값이 있다면
            if t["date"] != date: #for로 순회중인 거래내역 의 date값과 같지 않다면 컨티뉴 
                continue

        else: #date안에 값이없다면
            if start_date is not None and t["date"] < start_date: #시작기간이 있고 그게 순회중인 t의 date값보다 크다면 조건문탈출.
                continue
            if end_date is not None and t["date"] > end_date: #종료기간이 있고 그게 순회중인 t의 date값보다 작다면 조건문 탈출.
                continue

        if category is not None and t["category"] != category:
            continue

        if keyword is not None and keyword.lower() not in t["description"].lower():
            continue

        results.append(t)


    # 3) 날짜 오름차순, 같으면 id 오름차순으로 정렬한다
    #    왜  : 목록을 보여줄 때 시간순이 자연스럽고, 여러 건일 때 사용자가 고르기 쉬워진다.
    #    방법: results.sort(key=lambda t: (t["date"], t["id"]))
    #          key 에 튜플을 주면 앞의 값으로 먼저, 같으면 뒤의 값으로 정렬한다.

    results.sort(key=lambda t: (t["date"], t["id"]))



    # 4) 결과를 규약대로 돌려준다
    #    왜  : 0건이어도 "검색은 성공했고 결과가 없을 뿐"이므로 ok 는 True 다.
    #          여기서 False 를 주면 Gemini가 오류가 났다고 오해한다.
    #          count 를 따로 담는 이유는, Gemini가 "여러 건이니 되물어야겠다"를 즉시 판단하게 하기 위해서다.
    #    방법: {"ok": True, "data": {"count": len(results), "transactions": results}, "message": ...}
    #          건수가 0일 때와 아닐 때 message 문구를 다르게 하면 더 친절하다.
        # 4) 반환
    if results:
        message = f"조건에 맞는 거래 {len(results)}건을 찾았습니다."
    else:
        message = "조건에 맞는 거래가 없습니다."

    return {
        "ok": True,
        "data": {"count": len(results), "transactions": results},
        "message": message,
    }
    # count는 필터 조건에 해당되는 거래의 갯수
    # transactions는 필터 조건에 해당된 것들의 내용.



BudgetManager.search_transactions = search_transactions


# ---- 확인 ----
print(manager.search_transactions()["data"]["count"])                          # 전체 (4)
print(manager.search_transactions(date="2026-08-27")["data"]["count"])         # 2
print(manager.search_transactions(start_date="2026-08-01",
                                  end_date="2026-08-27")["data"]["count"])     # 3
print(manager.search_transactions(category="식비")["data"]["count"])            # 2
print(manager.search_transactions(keyword="카페")["data"]["count"])             # 1

print()
# 연속 호출 시연에 쓸 조건 — 반드시 1건이어야 한다
result = manager.search_transactions(date="2026-08-27", keyword="카페")
print(result["data"]["count"], result["data"]["transactions"])

print()
# 결과가 없는 조건 — ok는 True, count는 0
print(manager.search_transactions(date="2020-01-01"))

4
2
3
2
1

1 [{'id': 2, 'type': '지출', 'category': '식비', 'amount': 5000, 'description': '카페 아메리카노', 'date': '2026-08-27'}]

{'ok': True, 'data': {'count': 0, 'transactions': []}, 'message': '조건에 맞는 거래가 없습니다.'}


##### 💡 search_transactions — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def search_transactions(self, date=None, start_date=None, end_date=None,
                        category=None, keyword=None):
    # 1) 결과를 담을 리스트
    results = []

    # 2) 조건에 맞지 않으면 건너뛴다
    for t in self.transactions:
        if date is not None:
            if t["date"] != date:
                continue
        else:
            if start_date is not None and t["date"] < start_date:
                continue
            if end_date is not None and t["date"] > end_date:
                continue

        if category is not None and t["category"] != category:
            continue

        if keyword is not None and keyword.lower() not in t["description"].lower():
            continue

        results.append(t)

    # 3) 정렬
    results.sort(key=lambda t: (t["date"], t["id"]))

    # 4) 반환
    if results:
        message = f"조건에 맞는 거래 {len(results)}건을 찾았습니다."
    else:
        message = "조건에 맞는 거래가 없습니다."

    return {
        "ok": True,
        "data": {"count": len(results), "transactions": results},
        "message": message,
    }


BudgetManager.search_transactions = search_transactions
```

</details>

#### 2-3. `update_transaction` — 거래 수정

<details open>
<summary>설명 펼치기</summary>

```python
def update_transaction(self, transaction_id, amount=None, category=None,
                       description=None, date=None):
```

`transaction_id`만 기본값이 없다. **무엇을 수정할지 모르면 아무것도 할 수 없기 때문**이다.
`transaction_type`은 수정 항목에 없다. 수입을 지출로 바꾸는 건 사실상 다른 거래라서 삭제 후 재등록이 맞다.

##### ID로 거래 찾기

```python
target = None
for t in self.transactions:
    if t["id"] == transaction_id:
        target = t
        break
```

> Gemini가 ID를 `"3"`처럼 **문자열로 보낼 수 있다.** `3 == "3"`은 `False`라서 못 찾는다.
> 비교 전에 `int()`로 바꾸되, 변환이 실패할 수 있으니 `try`/`except`로 감싸고 실패하면 `NOT_FOUND`로 처리한다.

##### 찾은 거래를 고치면 원본이 바뀐다

리스트에서 꺼낸 딕셔너리는 **복사본이 아니라 원본을 가리키는 이름표**다.

```python
target["amount"] = 4500     # self.transactions 안의 그 거래도 함께 바뀐다
```

그래서 따로 다시 집어넣을 필요가 없다.

##### `before`는 반드시 `.copy()`

```python
before = target.copy()      # 지금 상태를 별도 딕셔너리로 복사
```

`before = target` 이라고 쓰면 **둘이 같은 딕셔너리를 가리켜서**, 수정하는 순간 `before`도 같이 바뀐다.
그러면 "5,000원 → 4,500원"이 아니라 "4,500원 → 4,500원"이 되어버린다.

##### ⭐ 검사를 먼저 다 하고, 그다음에 고친다

검사와 수정을 섞으면 **중간에 실패했을 때 앞부분만 바뀐 상태**로 남는다.
그래서 주어진 값들을 **먼저 전부 검사해서 임시 변수에 담아두고**, 통과한 뒤에 한꺼번에 반영한다.

##### 전부 `None`이면 거절한다

이 검사가 없으면 `update_transaction(3)` 처럼 불렀을 때 **아무 일도 안 했는데 성공했다고 답한다.**

##### 카테고리는 기존 유형 기준으로 검사

수정할 때는 `transaction_type`을 받지 않으므로, **찾은 거래의 기존 `type`**을 기준으로 삼는다.

```python
self._validate_category(category, target["type"])
```

</details>

In [11]:
def update_transaction(self, transaction_id, amount=None, category=None,
                       description=None, date=None):

    # 1) transaction_id 를 정수로 바꾼다
    #    왜  : Gemini가 ID를 "3" 처럼 문자열로 보낼 수 있다.
    #          파이썬에서 3 == "3" 은 False라서, 그대로 두면 있는 거래도 못 찾는다.
    #    방법: try: transaction_id = int(transaction_id)
    #          except (TypeError, ValueError): NOT_FOUND 를 return
    #          (숫자로 바꿀 수 없는 값이 왔다는 건 그런 거래가 없다는 뜻이므로 NOT_FOUND 로 처리한다)
    # 1) ID를 정수로
    try:
        transaction_id = int(transaction_id)
    except (TypeError, ValueError):
        return {"ok": False, "error": "NOT_FOUND",
                "message": f"거래 ID가 올바르지 않습니다: {transaction_id}"}


    # 2) 그 ID를 가진 거래를 찾는다
    #    왜  : 수정하려면 대상이 필요하다. 리스트에는 "3번 거래를 바로 꺼내는" 기능이 없으므로
    #          처음부터 하나씩 훑으면서 id 가 같은 것을 찾아야 한다.
    #    방법: target = None 으로 시작해서 for 문을 돌고,
    #          t["id"] == transaction_id 인 것을 만나면 target 에 담고 break.
    #          반복이 끝났는데도 target 이 None 이면 NOT_FOUND 를 return.
    target = None
    for t in self.transactions:
        if t["id"] == transaction_id:
            target = t
            break
    if target is None:
        return {"ok": False, "error": "NOT_FOUND",
                "message": f"{transaction_id}번 거래를 찾을 수 없습니다."}


    # 3) 수정할 항목이 하나도 없으면 거절한다
    #    왜  : 이 검사가 없으면 update_transaction(3) 처럼 불렀을 때
    #          아무것도 안 바꿨는데 "수정했습니다"라고 답한다. Gemini가 사용자에게 거짓말을 하게 된다.
    #    방법: amount, category, description, date 가 전부 None 이면
    #          NOTHING_TO_UPDATE 를 return.

    if amount is None and category is None and description is None and date is None:
        return {"ok": False, "error": "NOTHING_TO_UPDATE",
                "message": "수정할 항목이 없습니다. 금액, 카테고리, 설명, 날짜 중 하나 이상을 알려주세요."}


    # 4) 주어진 값들을 "먼저 전부" 검사한다  ⭐ 아직 고치지 않는다
    #    왜  : 검사와 수정을 섞으면, 중간에 실패했을 때 앞부분만 바뀐 상태로 남는다.
    #          (금액은 바뀌었는데 날짜는 안 바뀐 반쪽짜리 거래가 된다)
    #          그래서 통과 여부를 전부 확인한 뒤, 5)~6)에서 한꺼번에 반영한다.
    #    방법: - amount 가 주어졌으면 new_amount = self._validate_amount(amount)
    #            None 이면 INVALID_AMOUNT return
    #          - date 가 주어졌으면 self._validate_date(date) 가 False면 INVALID_DATE return
    #          - category 가 주어졌으면 self._validate_category(category, target["type"]) 로 검사
    #            ⚠️ 여기서는 target["type"] 을 쓴다. 수정할 때는 유형을 받지 않으므로
    #               "이 거래가 원래 무엇이었는지"를 기준으로 삼아야 한다.
    new_amount = None
    if amount is not None:
        new_amount = self._validate_amount(amount)
        if new_amount is None:
            return {"ok": False, "error": "INVALID_AMOUNT",
                    "message": "금액은 0보다 큰 숫자여야 합니다."}
        
    if date is not None and not self._validate_date(date):
        return {"ok": False, "error" : "INVALID_DATE", "message": "날짜는 YYYY-MM-DD 형식이어야합니다. 예: 2026-08-27"}

    if category is not None and not self._validate_category(category, target["type"]):
        if target["type"] =="지출":
            allowed = ", ".join(self.expense_categories)
        else:
            allowed = ", ".join(self.income_categories)
        return {"ok": False, "error": "INVALID_CATEGORY",
                "message": f"'{category}'는 {target['type']} 카테고리가 아닙니다. 사용 가능: {allowed}"}
    
    # 5) 고치기 전 상태를 복사해 둔다
    #    왜  : 리스트에서 꺼낸 딕셔너리는 복사본이 아니라 원본을 가리키는 이름표다.
    #          before = target 이라고 쓰면 둘이 같은 것을 가리켜서, 수정하는 순간 before 도 함께 바뀐다.
    #          그러면 "5,000원 → 4,500원"이 아니라 "4,500원 → 4,500원"이 되어버린다.
    #    방법: before = target.copy()
    #          (거래 딕셔너리는 안에 리스트가 없는 평평한 구조라 .copy() 로 충분하다)
    before = target.copy()


    # 6) 주어진 항목만 실제로 반영한다
    #    왜  : None 인 항목은 "이건 안 바꾼다"는 뜻이다. 무조건 대입하면 안 바꾸려던 값이 None 으로 지워진다.
    #    방법: if amount is not None: target["amount"] = new_amount   ... 이런 식으로 항목마다
    #          무엇이 어떻게 바뀌었는지 문장을 리스트에 모아두면 7)의 message 가 풍부해진다.
    #          예: changes.append(f"금액 {before['amount']:,}원 → {new_amount:,}원")
    changes = []
    if new_amount is not None:
        target["amount"] = new_amount
        changes.append(f"금액 {before['amount']:,}원 → {new_amount:,}원")
    if category is not None:
        target["category"] = category
        changes.append(f"카테고리 {before['category']} → {category}")
    if description is not None:
        target["description"] = description
        changes.append(f"설명 '{before['description']}' → '{description}'")
    if date is not None:
        target["date"] = date
        changes.append(f"날짜 {before['date']} → {date}")


    # 7) before / after 를 함께 돌려준다
    #    왜  : 둘 다 주면 Gemini가 "무엇이 어떻게 바뀌었는지" 정확히 설명할 수 있고,
    #          시연할 때도 변화가 눈에 보인다.
    #    방법: {"ok": True, "data": {"before": before, "after": target.copy()}, "message": ...}
    return {
        "ok": True,
        "data": {"before": before, "after": target.copy()},
        "message": f"[{transaction_id}]번 거래를 수정했습니다. " + ", ".join(changes),
    }


BudgetManager.update_transaction = update_transaction




# ---- 확인 ----
r = manager.update_transaction(2, amount=4500)
print(r["message"])
print("before:", r["data"]["before"]["amount"], "/ after:", r["data"]["after"]["amount"])
# 두 값이 같다면 .copy() 를 빠뜨린 것이다

print()
print(manager.update_transaction("3", description="버스로 변경")["message"])   # 문자열 ID도 되어야 한다
print(manager.update_transaction(999, amount=5000))                          # NOT_FOUND
print(manager.update_transaction(1))                                          # NOTHING_TO_UPDATE
print(manager.update_transaction(1, category="월급"))                          # INVALID_CATEGORY

print()
print(manager.search_transactions(date="2026-08-27")["data"]["transactions"])

[2]번 거래를 수정했습니다. 금액 5,000원 → 4,500원
before: 5000 / after: 4500

[3]번 거래를 수정했습니다. 설명 '지하철' → '버스로 변경'
{'ok': False, 'error': 'NOT_FOUND', 'message': '999번 거래를 찾을 수 없습니다.'}
{'ok': False, 'error': 'NOTHING_TO_UPDATE', 'message': '수정할 항목이 없습니다. 금액, 카테고리, 설명, 날짜 중 하나 이상을 알려주세요.'}
{'ok': False, 'error': 'INVALID_CATEGORY', 'message': "'월급'는 지출 카테고리가 아닙니다. 사용 가능: 식비, 교통비, 주거비, 문화생활, 기타"}

[{'id': 1, 'type': '지출', 'category': '식비', 'amount': 12000, 'description': '점심', 'date': '2026-08-27'}, {'id': 2, 'type': '지출', 'category': '식비', 'amount': 4500, 'description': '카페 아메리카노', 'date': '2026-08-27'}]


##### 💡 update_transaction — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def update_transaction(self, transaction_id, amount=None, category=None,
                       description=None, date=None):
    # 1) ID를 정수로
    try:
        transaction_id = int(transaction_id)
    except (TypeError, ValueError):
        return {"ok": False, "error": "NOT_FOUND",
                "message": f"거래 ID가 올바르지 않습니다: {transaction_id}"}

    # 2) 거래 찾기
    target = None
    for t in self.transactions:
        if t["id"] == transaction_id:
            target = t
            break

    if target is None:
        return {"ok": False, "error": "NOT_FOUND",
                "message": f"{transaction_id}번 거래를 찾을 수 없습니다."}

    # 3) 수정할 항목이 하나도 없으면
    if amount is None and category is None and description is None and date is None:
        return {"ok": False, "error": "NOTHING_TO_UPDATE",
                "message": "수정할 항목이 없습니다. 금액, 카테고리, 설명, 날짜 중 하나 이상을 알려주세요."}

    # 4) 먼저 전부 검사한다
    new_amount = None
    if amount is not None:
        new_amount = self._validate_amount(amount)
        if new_amount is None:
            return {"ok": False, "error": "INVALID_AMOUNT",
                    "message": "금액은 0보다 큰 숫자여야 합니다."}

    if date is not None and not self._validate_date(date):
        return {"ok": False, "error": "INVALID_DATE",
                "message": "날짜는 YYYY-MM-DD 형식이어야 합니다. 예: 2026-08-27"}

    if category is not None and not self._validate_category(category, target["type"]):
        if target["type"] == "지출":
            allowed = ", ".join(self.expense_categories)
        else:
            allowed = ", ".join(self.income_categories)
        return {"ok": False, "error": "INVALID_CATEGORY",
                "message": f"'{category}'는 {target['type']} 카테고리가 아닙니다. 사용 가능: {allowed}"}

    # 5) 고치기 전 상태를 복사
    before = target.copy()

    # 6) 반영
    changes = []
    if new_amount is not None:
        target["amount"] = new_amount
        changes.append(f"금액 {before['amount']:,}원 → {new_amount:,}원")
    if category is not None:
        target["category"] = category
        changes.append(f"카테고리 {before['category']} → {category}")
    if description is not None:
        target["description"] = description
        changes.append(f"설명 '{before['description']}' → '{description}'")
    if date is not None:
        target["date"] = date
        changes.append(f"날짜 {before['date']} → {date}")

    # 7) 반환
    return {
        "ok": True,
        "data": {"before": before, "after": target.copy()},
        "message": f"[{transaction_id}]번 거래를 수정했습니다. " + ", ".join(changes),
    }


BudgetManager.update_transaction = update_transaction
```

</details>

### Phase 3 — 예산 도구 2개

<details open>
<summary>설명 펼치기</summary>

#### 3-1. `set_budget` — 예산 설정

예산은 **딕셔너리 안에 딕셔너리**로 저장한다. 바깥 열쇠는 월, 안쪽 열쇠는 카테고리다.

```python
self.budgets = {
    "2026-08": {"식비": 300000, "교통비": 50000},
    "2026-09": {"식비": 400000}
}
```

카테고리 하나만 열쇠로 쓰면 8월과 9월에 다른 예산을 세울 수 없기 때문이다.

##### 예산은 지출에만 설정한다

```python
self._validate_category(category, "지출")     # 유형을 "지출"로 고정해서 검사
```

예산은 "이만큼만 쓰자"는 뜻이라 수입에는 의미가 없다. *"월급 예산 300만원"*은 말이 안 되므로 여기서 막는다.

##### 2단계 딕셔너리에 값 넣기

바깥 열쇠(월)가 **아직 없을 수 있다.** 없는 열쇠에 바로 접근하면 `KeyError`가 난다.

```python
self.budgets["2026-08"]["식비"] = 300000    # "2026-08"이 없으면 에러!
```

그래서 **없으면 먼저 빈 딕셔너리를 만들어 두고** 넣는다.

```python
if month not in self.budgets:
    self.budgets[month] = {}
self.budgets[month][category] = value
```

##### 이미 있으면 덮어쓴다

"식비 예산 30만원" 했다가 "역시 40만원"이라고 하면 40만원이 되는 게 자연스럽다.
별도 처리 없이 그냥 대입하면 된다.

</details>

In [12]:
def set_budget(self, month, category, amount):

    # 1) 월 형식을 검사한다
    #    왜  : 예산은 "2026-08" 이라는 열쇠로 저장된다. 형식이 제각각이면
    #          "2026-8" 로 저장한 예산을 "2026-08" 로 조회할 때 못 찾는다.
    #    방법: self._validate_month(month) 가 False면 INVALID_MONTH 를 return
    if not self._validate_month(month):
        return {"ok": False, "error": "INVALID_MONTH",
                "message": "월은 YYYY-MM 형식이어야 합니다. 예: 2026-08"}



    # 2) 지출 카테고리인지 검사한다
    #    왜  : 예산은 "이만큼만 쓰자"는 뜻이라 수입에는 의미가 없다.
    #          "월급 예산 300만원" 같은 말이 안 되는 요청을 여기서 막는다.
    #    방법: self._validate_category(category, "지출") 처럼 유형을 "지출"로 고정해서 부른다.
    #          False면 INVALID_CATEGORY 를 return 하되,
    #          message 에 "예산은 지출 카테고리에만 설정할 수 있습니다" 와 사용 가능 목록을 넣어준다.
    if not self._validate_category(category, "지출"):
        allowed = ", ".join(self.expense_categories)
        return {"ok": False, "error": "INVALID_CATEGORY",
                "message": f"예산은 지출 카테고리에만 설정할 수 있습니다. 사용 가능: {allowed}"}


    # 3) 금액을 검사하면서 정수로 바꾼다
    #    왜  : add_transaction 과 같은 이유다. 나중에 remaining = budget - spent 계산을 하려면
    #          예산도 정수여야 한다.
    #    방법: value = self._validate_amount(amount), None 이면 INVALID_AMOUNT return
    value = self._validate_amount(amount)
    if value is None:
        return {"ok": False, "error": "INVALID_AMOUNT",
                "message": "예산은 0보다 큰 숫자여야 합니다."}


    # 4) budgets 에 저장한다
    #    왜  : self.budgets 는 {"2026-08": {"식비": 300000}} 처럼 딕셔너리 안에 딕셔너리다.
    #          바깥 열쇠(월)가 아직 없는 상태에서 self.budgets[month][category] 로 바로 접근하면
    #          KeyError 가 난다. 그래서 없으면 빈 딕셔너리를 먼저 만들어 두어야 한다.
    #    방법: if month not in self.budgets:
    #              self.budgets[month] = {}
    #          self.budgets[month][category] = value
    #          (이미 값이 있으면 그냥 덮어쓰면 된다. "역시 40만원으로" 가 자연스럽게 처리된다)
    if month not in self.budgets:
        self.budgets[month] = {}
    self.budgets[month][category] = value


    # 5) 성공 결과를 돌려준다
    #    방법: {"ok": True, "data": {"month": ..., "category": ..., "amount": value}, "message": ...}
    return {
        "ok": True,
        "data": {"month": month, "category": category, "amount": value},
        "message": f"{month} {category} 예산을 {value:,}원으로 설정했습니다.",
    }



BudgetManager.set_budget = set_budget


# ---- 확인 ----
print(manager.set_budget("2026-08", "식비", 300000)["message"])
print(manager.set_budget("2026-08", "식비", 350000)["message"])   # 덮어쓰기
print(manager.set_budget("2026-08", "교통비", 50000)["message"])
print(manager.set_budget("2026-09", "식비", 400000)["message"])

print()
print(manager.budgets)

print()
print(manager.set_budget("2026-08", "월급", 3000000))   # INVALID_CATEGORY (예산은 지출만)
print(manager.set_budget("2026-13", "식비", 300000))    # INVALID_MONTH  (13월은 없다)
print(manager.set_budget("2026-08", "식비", -100))      # INVALID_AMOUNT

2026-08 식비 예산을 300,000원으로 설정했습니다.
2026-08 식비 예산을 350,000원으로 설정했습니다.
2026-08 교통비 예산을 50,000원으로 설정했습니다.
2026-09 식비 예산을 400,000원으로 설정했습니다.

{'2026-08': {'식비': 350000, '교통비': 50000}, '2026-09': {'식비': 400000}}

{'ok': False, 'error': 'INVALID_CATEGORY', 'message': '예산은 지출 카테고리에만 설정할 수 있습니다. 사용 가능: 식비, 교통비, 주거비, 문화생활, 기타'}
{'ok': False, 'error': 'INVALID_MONTH', 'message': '월은 YYYY-MM 형식이어야 합니다. 예: 2026-08'}
{'ok': False, 'error': 'INVALID_AMOUNT', 'message': '예산은 0보다 큰 숫자여야 합니다.'}


##### 💡 set_budget — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def set_budget(self, month, category, amount):
    # 1) 월 형식
    if not self._validate_month(month):
        return {"ok": False, "error": "INVALID_MONTH",
                "message": "월은 YYYY-MM 형식이어야 합니다. 예: 2026-08"}

    # 2) 지출 카테고리인지
    if not self._validate_category(category, "지출"):
        allowed = ", ".join(self.expense_categories)
        return {"ok": False, "error": "INVALID_CATEGORY",
                "message": f"예산은 지출 카테고리에만 설정할 수 있습니다. 사용 가능: {allowed}"}

    # 3) 금액
    value = self._validate_amount(amount)
    if value is None:
        return {"ok": False, "error": "INVALID_AMOUNT",
                "message": "예산은 0보다 큰 숫자여야 합니다."}

    # 4) 저장 (바깥 열쇠가 없으면 먼저 만든다)
    if month not in self.budgets:
        self.budgets[month] = {}
    self.budgets[month][category] = value

    # 5) 성공
    return {
        "ok": True,
        "data": {"month": month, "category": category, "amount": value},
        "message": f"{month} {category} 예산을 {value:,}원으로 설정했습니다.",
    }


BudgetManager.set_budget = set_budget
```

</details>

#### 3-2. `get_budget_status` — 사용액과 남은 예산 조회

<details open>
<summary>설명 펼치기</summary>

"이번 달 식비 얼마나 남았어?"에 답하려면 두 가지가 필요하다.

- **예산** — `self.budgets`에 저장해둔 값
- **사용액** — 그 달, 그 카테고리의 **지출 거래를 전부 더한 값**

사용액은 저장해두지 않고 **부를 때마다 계산한다.** 저장해두면 거래가 수정·삭제될 때마다 같이 고쳐야 해서 어긋나기 쉽다.

##### 해당 월 거래 고르기

```python
"2026-08-27".startswith("2026-08")    # True
```

`startswith`는 "이 문자열로 시작하는가"를 묻는다. 날짜 형식을 고정해둔 덕분에 이렇게 간단히 해결된다.

##### ⚠️ 반드시 지출만 더한다

```python
if t["type"] == "지출" and t["category"] == cat and t["date"].startswith(month):
```

`type` 검사를 빠뜨리면 8월 월급 300만원이 식비 사용액에 더해져서 *"예산을 293만원 초과했습니다"* 같은 답이 나온다.
**이 프로젝트에서 가장 실수하기 쉬운 지점이다.**

##### 계산할 값

| 이름 | 계산 |
|---|---|
| `spent` | 조건에 맞는 거래들의 `amount` 합계 |
| `remaining` | `budget - spent` (**음수 가능** — 초과 지출) |
| `usage_rate` | `round(spent / budget * 100, 1)` |
| `over_budget` | `spent > budget` |

`remaining`을 0으로 막으면 "얼마나 초과했는지"를 알려줄 수 없다.
`usage_rate`를 `round(값, 1)`로 정리하지 않으면 `37.333333333333336` 같은 값이 나온다.

##### 예산이 없으면 오류

- 그 달에 예산이 하나도 없으면 → `NO_BUDGET_SET`
- `category`를 지정했는데 그 예산이 없으면 → `NO_BUDGET_SET`

예산을 안 세웠는데 "남은 금액"을 말할 수는 없다. 조용히 0으로 처리하지 않는다.

##### 반환은 리스트로

카테고리가 여러 개일 수 있으므로 `items` 리스트에 담는다.
하나만 조회해도 리스트 안에 하나가 든 형태로 통일하면, Gemini가 결과를 읽는 방법이 항상 같아진다.

</details>

In [ ]:
def get_budget_status(self, month, category=None):

    # 1) 월 형식을 검사한다
    #    방법: self._validate_month(month) 가 False면 INVALID_MONTH return



    # 2) 그 달의 예산을 꺼낸다
    #    왜  : 예산을 세우지 않았는데 "남은 금액"을 말할 수는 없다.
    #          조용히 0으로 처리하면 사용자가 예산을 세운 줄 착각한다. 오류로 분명히 알린다.
    #    방법: month_budgets = self.budgets.get(month)
    #          .get 을 쓰면 열쇠가 없어도 KeyError 대신 None 이 나온다.
    #          None 이거나 비어 있으면 NO_BUDGET_SET 을 return



    # 3) 계산할 카테고리 목록을 정한다
    #    왜  : category 를 주면 하나만, 안 주면 그 달 전부를 계산해야 한다.
    #          두 경우를 각각 다르게 처리하면 코드가 두 벌이 된다.
    #          "대상 목록"으로 통일해두면 아래 반복문 하나로 둘 다 처리된다.
    #    방법: category 가 주어졌으면
    #              그 카테고리가 month_budgets 에 없으면 NO_BUDGET_SET return
    #              있으면 targets = [category]      ← 하나짜리 리스트로 만든다
    #          category 가 없으면
    #              targets = list(month_budgets.keys())



    # 4) 카테고리마다 사용액을 계산해서 items 리스트에 담는다
    #    왜  : 사용액은 따로 저장해두지 않고 부를 때마다 계산한다.
    #          저장해두면 거래가 수정·삭제될 때마다 같이 고쳐야 해서 금방 어긋난다.
    #          매번 세면 항상 정확하다.
    #    방법: for cat in targets: 안에서
    #          budget = month_budgets[cat]
    #          spent = 0 으로 시작해, 아래 세 조건을 모두 만족하는 거래의 amount 를 더한다
    #            ① t["type"] == "지출"                 ⚠️ 이걸 빠뜨리면 월급이 식비 사용액에 더해진다
    #            ② t["category"] == cat
    #            ③ t["date"].startswith(month)         "2026-08-27".startswith("2026-08") → True
    #          remaining   = budget - spent            음수를 그대로 둔다 (얼마나 초과했는지 알려주려면 필요)
    #          usage_rate  = round(spent / budget * 100, 1)   round 를 안 하면 37.33333... 이 나온다
    #          over_budget = spent > budget
    #          이 값들을 딕셔너리로 만들어 items 에 append



    # 5) 결과를 돌려준다
    #    왜  : 카테고리가 여러 개일 수 있으므로 items 리스트에 담는다.
    #          하나만 조회해도 리스트 안에 하나가 든 형태로 통일하면,
    #          Gemini가 결과를 읽는 방법이 항상 같아진다.
    #    방법: {"ok": True, "data": {"month": month, "items": items}, "message": ...}
    #          message 는 items 를 돌면서 한 줄씩 만들어 이어붙이면 된다.
    #          초과한 경우와 아닌 경우 문구를 다르게 하면 더 친절하다.



BudgetManager.get_budget_status = get_budget_status


# ---- 확인 ----
r = manager.get_budget_status("2026-08", "식비")
print(r["message"])
print(r["data"]["items"])
# spent 는 8월 식비 지출의 합이어야 한다. 월급 3,000,000원이 섞이면 잘못된 것이다.

print()
print(manager.get_budget_status("2026-08")["data"]["items"])   # 전체 카테고리

print()
manager.add_transaction("지출", "식비", 500000, "2026-08-29", "회식")
r = manager.get_budget_status("2026-08", "식비")
print(r["message"])
print("초과 여부:", r["data"]["items"][0]["over_budget"])       # True 여야 한다

print()
print(manager.get_budget_status("2026-12"))                    # NO_BUDGET_SET
print(manager.get_budget_status("2026-08", "주거비"))           # NO_BUDGET_SET

##### 💡 get_budget_status — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def get_budget_status(self, month, category=None):
    # 1) 월 형식
    if not self._validate_month(month):
        return {"ok": False, "error": "INVALID_MONTH",
                "message": "월은 YYYY-MM 형식이어야 합니다. 예: 2026-08"}

    # 2) 그 달 예산 꺼내기
    month_budgets = self.budgets.get(month)
    if not month_budgets:
        return {"ok": False, "error": "NO_BUDGET_SET",
                "message": f"{month}에 설정된 예산이 없습니다."}

    # 3) 계산 대상 정하기
    if category is not None:
        if category not in month_budgets:
            return {"ok": False, "error": "NO_BUDGET_SET",
                    "message": f"{month} {category} 예산이 설정되어 있지 않습니다."}
        targets = [category]
    else:
        targets = list(month_budgets.keys())

    # 4) 카테고리별 계산
    items = []
    for cat in targets:
        budget = month_budgets[cat]

        spent = 0
        for t in self.transactions:
            if (t["type"] == "지출"
                    and t["category"] == cat
                    and t["date"].startswith(month)):
                spent += t["amount"]

        items.append({
            "category": cat,
            "budget": budget,
            "spent": spent,
            "remaining": budget - spent,
            "usage_rate": round(spent / budget * 100, 1),
            "over_budget": spent > budget,
        })

    # 5) 안내 문장 만들기
    lines = []
    for item in items:
        if item["over_budget"]:
            lines.append(f"{item['category']}: 예산 {item['budget']:,}원 중 "
                         f"{item['spent']:,}원 사용, {-item['remaining']:,}원 초과")
        else:
            lines.append(f"{item['category']}: 예산 {item['budget']:,}원 중 "
                         f"{item['spent']:,}원 사용, {item['remaining']:,}원 남음")

    return {
        "ok": True,
        "data": {"month": month, "items": items},
        "message": f"{month} 예산 현황 — " + " / ".join(lines),
    }


BudgetManager.get_budget_status = get_budget_status
```

</details>

### Phase 4 — 저장과 불러오기

<details open>
<summary>설명 펼치기</summary>

지금까지 만든 데이터는 전부 **메모리에만** 있다. 커널을 다시 시작하면 사라진다.
파일에 적어두고 시작할 때 다시 읽어오면 **재실행해도 데이터가 남는다.**

##### JSON이란

여러 언어가 공통으로 읽고 쓸 수 있는 **텍스트 형식**이다. 생김새가 파이썬 딕셔너리·리스트와 거의 같다.

```json
{"id": 1, "type": "지출", "amount": 12000}
```

우리가 데이터를 딕셔너리와 리스트로만 설계한 덕분에 **변환 코드 없이 그대로 저장된다.**

##### 파일에 쓰기

```python
with open(self.data_file, "w", encoding="utf-8") as f:
    json.dump(데이터, f, ensure_ascii=False, indent=2)
```

| 부분 | 의미 |
|---|---|
| `"w"` | 쓰기(write) 모드. 기존 내용은 지워지고 새로 쓴다 |
| `encoding="utf-8"` | 한글이 깨지지 않게 하는 설정. **윈도우에서는 빠뜨리면 거의 반드시 깨진다** |
| `with ... as f:` | 블록이 끝나면 **파일을 자동으로 닫아준다** |
| `ensure_ascii=False` | 한글을 `\uc2dd\ube44` 대신 `식비` 그대로 저장한다 |
| `indent=2` | 줄바꿈과 들여쓰기를 넣어 사람이 읽을 수 있게 한다 |

**`with`를 쓰는 이유:** 직접 `f.close()`를 부르면 중간에 에러가 났을 때 닫지 못한다.
`with`는 에러가 나든 안 나든 블록을 벗어날 때 반드시 닫아준다.

##### `dump` / `load` / `dumps` / `loads`

| 함수 | 하는 일 |
|---|---|
| `json.dump(데이터, f)` | 파이썬 → **파일**에 쓰기 |
| `json.load(f)` | **파일** → 파이썬으로 읽기 |
| `json.dumps(데이터)` | 파이썬 → **문자열** (`s`는 string) |
| `json.loads(문자열)` | **문자열** → 파이썬 |

`s`가 붙으면 파일이 아니라 문자열을 다룬다. Phase 6에서 `json.dumps`를 쓰게 된다.

##### ⚠️ `next_id`를 반드시 저장한다

빠뜨리면 재실행할 때 ID가 다시 1부터 시작해서 **기존 거래와 번호가 겹친다.**
그러면 "3번 수정해줘"가 엉뚱한 거래를 고친다.

##### 읽을 때는 파일이 없을 수 있다

**첫 실행에는 파일이 아직 없다.** 그냥 열면 `FileNotFoundError`로 죽는다.

```python
try:
    with open(self.data_file, "r", encoding="utf-8") as f:
        data = json.load(f)
except FileNotFoundError:
    return          # 파일이 없으면 기본값 그대로 두고 끝낸다
```

읽은 뒤 값을 담을 때는 `data.get("열쇠", 기본값)`을 쓰면 **파일에 그 항목이 없어도 에러가 나지 않는다.**

</details>

In [ ]:
def _save(self):

    # 1) 저장할 내용을 딕셔너리 하나로 모은다
    #    왜  : 지금 데이터는 self.transactions, self.next_id, ... 로 흩어져 있다.
    #          JSON 파일 하나에 담으려면 먼저 한 덩어리로 묶어야 한다.
    #    방법: 키 5개를 가진 딕셔너리를 만든다.
    #          "transactions" / "next_id" / "expense_categories" / "income_categories" / "budgets"
    #          ⚠️ next_id 를 빠뜨리면 재실행할 때 ID가 1부터 다시 시작해서
    #             기존 거래와 번호가 겹치고, "3번 수정해줘"가 엉뚱한 거래를 고친다.



    # 2) 파일에 쓴다
    #    왜  : with 를 쓰면 도중에 에러가 나도 파일이 반드시 닫힌다.
    #          encoding 과 ensure_ascii 를 빠뜨리면 한글이 깨지거나 \uc2dd\ube44 처럼 저장된다.
    #    방법: with open(self.data_file, "w", encoding="utf-8") as f:
    #              json.dump(데이터, f, ensure_ascii=False, indent=2)
    #          "w" 는 쓰기 모드다. 기존 내용은 지워지고 새로 쓴다.



def _load(self):

    # 1) 파일을 읽는다 — 없을 수도 있다
    #    왜  : 첫 실행에는 파일이 아직 없다. 그냥 열면 FileNotFoundError 로 프로그램이 죽는다.
    #          파일이 없는 건 오류가 아니라 "아직 저장한 게 없는 정상 상황"이므로,
    #          조용히 기본값 그대로 두고 끝내야 한다.
    #    방법: try: 안에서 "r"(읽기) 모드로 열고 data = json.load(f)
    #          except FileNotFoundError: 이면 아무것도 하지 않고 return



    # 2) 읽어온 값을 self 에 담는다
    #    왜  : json.load 로 얻은 건 그냥 지역 변수다. self. 에 담아야 인스턴스가 기억한다.
    #    방법: self.transactions = data.get("transactions", [])  ... 이런 식으로 5개
    #          .get(열쇠, 기본값) 을 쓰면 예전에 저장한 파일에 그 항목이 없어도 에러가 나지 않는다.
    #          카테고리 목록은 기본값으로 지금 값(self.expense_categories)을 주면 안전하다.



BudgetManager._save = _save
BudgetManager._load = _load


# ---- 확인 ----
manager._save()
print("저장 완료")

with open("data.json", "r", encoding="utf-8") as f:
    print(f.read()[:300])

##### 💡 _save / _load — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def _save(self):
    data = {
        "transactions": self.transactions,
        "next_id": self.next_id,
        "expense_categories": self.expense_categories,
        "income_categories": self.income_categories,
        "budgets": self.budgets,
    }
    with open(self.data_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def _load(self):
    try:
        with open(self.data_file, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        return          # 첫 실행이라 파일이 없다 → 기본값 그대로

    self.transactions = data.get("transactions", [])
    self.next_id = data.get("next_id", 1)
    self.expense_categories = data.get("expense_categories", self.expense_categories)
    self.income_categories = data.get("income_categories", self.income_categories)
    self.budgets = data.get("budgets", {})


BudgetManager._save = _save
BudgetManager._load = _load
```

</details>

#### 4-2. 자동 저장 연결하기

<details open>
<summary>설명 펼치기</summary>

이제 **데이터를 바꾸는 도구들이 끝날 때마다** 저장하도록 연결한다.

##### ① 세 함수에 한 줄씩 추가

`add_transaction`, `update_transaction`, `set_budget` 셀로 올라가서,
**성공 반환(`return {"ok": True, ...}`) 바로 앞**에 이 줄을 넣는다.

```python
    self._save()
```

검색과 조회는 데이터를 바꾸지 않으므로 넣지 않는다.

##### ② `__init__`의 주석 풀기

클래스 셀의 `__init__` 마지막 줄 주석을 푼다.

```python
        self._load()
```

이러면 `BudgetManager()`를 만드는 순간 저장된 데이터가 자동으로 복원된다.

##### ③ 그다음 Run All

클래스 셀을 다시 실행하면 **붙여둔 메서드들이 전부 사라진다.**
그러니 여기서는 **Run All(모두 실행)**로 처음부터 다시 돌리는 게 확실하다.

> ⚠️ 이미 `data.json`에 테스트 데이터가 쌓여 있다면, Run All을 할 때 그 위에 또 등록된다.
> 깨끗하게 시작하고 싶으면 폴더에서 `data.json`을 지우고 Run All 하면 된다.

</details>

In [ ]:
# 자동 저장이 동작하는지 확인한다.
# 핵심: "새 인스턴스를 만들었을 때 이전 데이터가 살아있는가"

# 1) 완전히 새로운 인스턴스를 만든다



# 2) 이전 데이터가 들어있는지 확인한다
#    - 거래 개수
#    - next_id 가 1로 리셋되지 않았는지   ⚠️ 가장 중요
#    - 예산



# 3) 새 인스턴스로 거래를 하나 더 등록해서, ID가 기존 번호에 이어서 붙는지 확인한다

##### 💡 자동 저장 확인 — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
# 1) 새 인스턴스 (파일에서 자동 복원된다)
manager2 = BudgetManager()

# 2) 이전 데이터 확인
print("거래 개수 :", len(manager2.transactions))
print("다음 ID   :", manager2.next_id)      # 1이면 안 된다
print("예산      :", manager2.budgets)

# 3) 이어서 등록되는지
print(manager2.add_transaction("지출", "문화생활", 15000, "2026-08-30", "영화")["message"])
```

</details>

### Phase 5 — Gemini에 도구로 등록하기

<details open>
<summary>설명 펼치기</summary>

여기서부터 Gemini가 등장한다. **지금까지 만든 함수는 하나도 고치지 않는다.**
바뀌는 건 딱 하나 — 함수를 부르는 주체가 "나"에서 "Gemini"로 바뀐다.

##### Function Calling — 모델은 함수를 실행하지 못한다

이름 때문에 오해하기 쉬운데, **Gemini가 우리 함수를 직접 실행하는 게 아니다.**
Gemini가 하는 일은 하나뿐이다. **"이 함수를 이런 인자로 불러주세요"라고 요청**하는 것.

```
사용자 질문
  → 모델: "add_transaction을 이런 인자로 불러줘"   (function_call)
  → 우리 프로그램: 실제로 파이썬 함수 실행
  → 모델에게 결과 전달                            (function_result)
  → 모델: 결과를 읽고 사용자에게 보여줄 문장 작성
```

실행 권한이 항상 우리 쪽에 있으므로, 모델이 요청해도 **등록하지 않은 함수는 절대 실행되지 않는다.**

##### 왜 결과를 다시 모델에게 보내나

우리 함수의 반환값 `{"ok": True, "data": {...}}` 은 **기계용 데이터**다.
모델은 그걸 읽고 *"12,000원 지출을 식비로 등록했어요"* 같은 자연스러운 문장으로 바꿔준다.

##### ⚠️ 여기서부터 Run All 금지

Gemini에 실제 요청이 나가서 **비용이 발생한다.** 필요한 셀만 골라서 실행한다.

##### 클라이언트 만들기

```python
client = genai.Client(api_key=api_key)
```

Gemini와 통신할 창구다. 앞으로 모든 요청을 이 `client`로 보낸다.

```python
model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
```

`os.getenv`의 **두 번째 인자는 기본값**이다. `.env`에 `GEMINI_MODEL`이 없어도 프로그램이 멈추지 않는다.

</details>

In [ ]:
# 1) genai 를 가져온다
#    왜  : Gemini와 통신하는 기능은 파이썬 기본 창고가 아니라 google 패키지에 들어 있다.
#    방법: from google import genai



# 2) API 키를 읽고, 없으면 즉시 멈춘다
#    왜  : 키가 없으면 앞으로의 모든 요청이 실패한다. 그때 나오는 에러 메시지는 알아보기 어려워서
#          원인을 찾느라 한참 헤매게 된다. 여기서 분명한 문장으로 미리 알려주는 편이 낫다.
#    방법: api_key = os.getenv("GEMINI_API_KEY")
#          if not api_key: raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")
#          raise 는 "에러를 일부러 일으켜서 프로그램을 멈춘다"는 명령이다.



# 3) 모델 이름을 읽는다
#    왜  : 모델을 바꾸고 싶을 때 코드를 고치지 않고 .env 만 바꾸면 되게 하려는 것이다.
#    방법: model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
#          두 번째 인자가 기본값이라, .env 에 없어도 프로그램이 멈추지 않는다.



# 4) 클라이언트를 만든다
#    왜  : Gemini와 통신할 창구다. 앞으로 모든 요청을 이 client 로 보낸다.
#          한 번만 만들어 두고 계속 재사용한다.
#    방법: client = genai.Client(api_key=api_key)



# 5) 준비됐는지 확인 출력
#    왜  : 어떤 모델로 돌아가는지 눈으로 확인해두면, 나중에 답변이 이상할 때 모델 문제인지 알 수 있다.

##### 💡 Gemini 클라이언트 준비 — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
from google import genai

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
client = genai.Client(api_key=api_key)

print("준비 완료 / 사용 모델:", model)
```

</details>

#### 5-2. 도구 schema 작성

<details open>
<summary>설명 펼치기</summary>

schema는 **모델에게 주는 함수 설명서**다. 모델은 우리 코드를 볼 수 없고 이 설명서만 읽는다.
설명이 부실하면 엉뚱한 도구를 고르거나 이상한 인자를 만들어낸다. **schema 품질이 곧 Agent 품질이다.**

##### 형태

```python
add_transaction_tool = {
    "type": "function",
    "name": "add_transaction",
    "description": "언제 이 도구를 쓰는지",
    "parameters": {
        "type": "object",
        "properties": {
            "매개변수이름": {"type": "string", "description": "무엇인지, 어떤 형식인지"},
        },
        "required": ["필수인 것만"],
    },
}
```

| 열쇠 | 의미 |
|---|---|
| `"type": "function"` | 우리가 만든 함수라는 표시 |
| `name` | ⚠️ **파이썬 함수 이름과 정확히 같아야 한다** |
| `description` | 모델이 "이 도구를 쓸까?"를 판단하는 근거 |
| `required` | 반드시 채워야 하는 매개변수 (= **기본값이 없는 것들**) |

`type`에 쓰는 값: `"string"` (문자열) / `"integer"` (정수) / `"number"` (실수) / `"boolean"`

##### `description`은 "언제 쓰는지"를 써야 한다

```python
"거래를 등록합니다."                                                    # 부족
"사용자가 돈을 쓰거나 벌었다고 말할 때 거래를 등록합니다. 예: '점심 12000원 썼어'"   # 좋다
```

**예시 문장을 넣는 게 특히 효과적이다.** 모델은 사용자 발화와 이 예시를 비교한다.

##### 매개변수에는 형식과 예시를

```python
"date": {"type": "string", "description": "거래 날짜. YYYY-MM-DD 형식. 예: 2026-08-27"}
```

형식을 안 적으면 모델이 `"8월 27일"`을 보내서 `_validate_date`에 걸린다.

##### `enum`으로 값을 제한한다

```python
"transaction_type": {"type": "string", "enum": ["수입", "지출"], "description": "..."}
```

모델이 `"소비"`나 `"expense"`를 지어내는 걸 막는다.

##### ⭐ 연속 도구 호출을 유도하는 한 문장

`update_transaction`의 설명에 **순서를 알려주는 문장**을 넣는다.

```python
"description": ("거래 ID를 알고 있을 때 그 거래를 수정합니다. "
                "ID를 모르면 먼저 search_transactions로 찾아 ID를 확인한 뒤 호출하세요.")
```

이 한 문장이 명세서 필수 시연인 *"검색 후 수정"*을 안정적으로 일어나게 만든다.

</details>

In [ ]:
# 도구 schema 는 "모델에게 주는 함수 설명서"다.
# 모델은 우리 코드를 볼 수 없고 이 설명서만 읽고 판단하므로, 여기가 부실하면 Agent 전체가 부실해진다.
#
# 공통 규칙
#   - "name" 은 파이썬 함수 이름과 글자 하나까지 똑같아야 한다 (모델이 이 문자열을 돌려주고, 그걸로 함수를 찾는다)
#   - "description" 에는 "무엇을 하는지"가 아니라 "언제 쓰는지"를 쓴다. 사용자 발화 예시를 넣으면 특히 잘 먹는다
#   - 각 매개변수 description 에는 형식과 예시를 적는다 ("YYYY-MM-DD 형식. 예: 2026-08-27")
#   - "required" 에는 파이썬 함수에서 기본값이 없는 매개변수만 넣는다
#     (required 에 넣으면 모델이 반드시 채워야 하고, 모르면 사용자에게 되묻는다)


# 1) add_transaction_tool
#    왜  : 등록은 값이 하나라도 틀리면 잘못된 데이터가 쌓인다. 그래서 설명을 가장 촘촘히 쓴다.
#    방법: transaction_type 에는 "enum": ["수입", "지출"] 을 넣어 값을 두 개로 제한한다.
#          (enum 이 없으면 모델이 "소비"나 "expense" 같은 걸 지어낸다)
#          category description 에는 지출/수입 목록을 둘 다 적어준다.
#          required: transaction_type, category, amount, date   (description 은 기본값이 있으므로 제외)



# 2) search_transactions_tool
#    왜  : 사용자는 조건 일부만 말한다. 어떤 조건들이 가능한지 모델이 알아야 골라 쓸 수 있다.
#    방법: 매개변수 5개(date, start_date, end_date, category, keyword)를 모두 적되,
#          파이썬에서 전부 기본값이 있으므로 required 는 빈 리스트 [] 로 둔다.
#          description 에 "8월에 식비로 쓴 거 보여줘" 같은 예시를 넣으면 기간 변환을 잘 한다.



# 3) update_transaction_tool
#    왜  : ⭐ 명세서 필수 시연인 "검색 후 수정"이 여기서 결정된다.
#          모델이 "ID를 모르면 먼저 검색해야 한다"는 순서를 알아야 도구를 두 번 연속 부른다.
#    방법: description 에 반드시 이런 문장을 넣는다.
#          "ID를 모르면 먼저 search_transactions로 거래를 찾아 ID를 확인한 뒤 호출하세요."
#          required: transaction_id 하나뿐 (나머지는 바꿀 항목만 넣으면 되므로)



# 4) set_budget_tool
#    방법: required: month, category, amount
#          description 에 "예산은 지출에만 설정할 수 있습니다" 를 넣어두면
#          모델이 수입 카테고리로 시도하는 것을 줄일 수 있다.



# 5) get_budget_status_tool
#    방법: required: month 만. category 는 선택이라 빼면 전체 카테고리를 조회한다.



# 6) TOOLS 리스트에 5개를 담고, 이름들을 출력해 확인한다
#    왜  : 요청을 보낼 때 tools=TOOLS 로 통째로 넘긴다. 하나라도 빠지면 그 기능은 Agent가 쓸 수 없다.

##### 💡 도구 schema — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
EXPENSE = "식비, 교통비, 주거비, 문화생활, 기타"
INCOME = "월급, 용돈, 부수입, 기타"

add_transaction_tool = {
    "type": "function",
    "name": "add_transaction",
    "description": ("사용자가 돈을 쓰거나 벌었다고 말할 때 수입 또는 지출 거래를 새로 등록합니다. "
                    "예: '점심 12000원 썼어', '월급 300만원 들어왔어'"),
    "parameters": {
        "type": "object",
        "properties": {
            "transaction_type": {
                "type": "string",
                "enum": ["수입", "지출"],
                "description": "거래 유형. 돈이 나갔으면 '지출', 들어왔으면 '수입'",
            },
            "category": {
                "type": "string",
                "description": f"카테고리. 지출: {EXPENSE} / 수입: {INCOME}",
            },
            "amount": {"type": "integer", "description": "금액(원). 0보다 큰 정수"},
            "date": {"type": "string", "description": "거래 날짜. YYYY-MM-DD 형식. 예: 2026-08-27"},
            "description": {"type": "string", "description": "거래 내용 메모. 예: 점심, 카페, 택시"},
        },
        "required": ["transaction_type", "category", "amount", "date"],
    },
}

search_transactions_tool = {
    "type": "function",
    "name": "search_transactions",
    "description": ("날짜, 기간, 카테고리, 설명 키워드로 거래를 검색합니다. "
                    "조건은 원하는 것만 넣으면 되고, 여러 조건을 함께 쓰면 모두 만족하는 거래를 찾습니다. "
                    "예: '8월에 식비로 쓴 거 보여줘', '카페에서 쓴 거 찾아줘'"),
    "parameters": {
        "type": "object",
        "properties": {
            "date": {"type": "string", "description": "특정 날짜. YYYY-MM-DD. 예: 2026-08-27"},
            "start_date": {"type": "string", "description": "기간 시작일. YYYY-MM-DD"},
            "end_date": {"type": "string", "description": "기간 종료일. YYYY-MM-DD"},
            "category": {"type": "string", "description": f"카테고리. 지출: {EXPENSE} / 수입: {INCOME}"},
            "keyword": {"type": "string", "description": "거래 설명에 포함된 단어. 예: 카페, 점심"},
        },
        "required": [],
    },
}

update_transaction_tool = {
    "type": "function",
    "name": "update_transaction",
    "description": ("거래 ID를 알고 있을 때 그 거래의 금액, 카테고리, 설명, 날짜를 수정합니다. "
                    "ID를 모르면 먼저 search_transactions로 거래를 찾아 ID를 확인한 뒤 호출하세요. "
                    "바꿀 항목만 넣으면 됩니다."),
    "parameters": {
        "type": "object",
        "properties": {
            "transaction_id": {"type": "integer", "description": "수정할 거래의 ID"},
            "amount": {"type": "integer", "description": "새 금액(원)"},
            "category": {"type": "string", "description": f"새 카테고리. 지출: {EXPENSE} / 수입: {INCOME}"},
            "description": {"type": "string", "description": "새 설명"},
            "date": {"type": "string", "description": "새 날짜. YYYY-MM-DD"},
        },
        "required": ["transaction_id"],
    },
}

set_budget_tool = {
    "type": "function",
    "name": "set_budget",
    "description": ("특정 월의 지출 카테고리 예산을 설정합니다. 예산은 지출에만 설정할 수 있습니다. "
                    "예: '이번 달 식비 예산을 30만원으로 설정해줘'"),
    "parameters": {
        "type": "object",
        "properties": {
            "month": {"type": "string", "description": "대상 월. YYYY-MM 형식. 예: 2026-08"},
            "category": {"type": "string", "description": f"지출 카테고리. {EXPENSE}"},
            "amount": {"type": "integer", "description": "예산 금액(원)"},
        },
        "required": ["month", "category", "amount"],
    },
}

get_budget_status_tool = {
    "type": "function",
    "name": "get_budget_status",
    "description": ("특정 월의 예산, 사용 금액, 남은 금액을 조회합니다. "
                    "카테고리를 지정하지 않으면 예산이 설정된 모든 카테고리를 알려줍니다. "
                    "예: '이번 달 식비 얼마나 남았어?'"),
    "parameters": {
        "type": "object",
        "properties": {
            "month": {"type": "string", "description": "조회할 월. YYYY-MM 형식. 예: 2026-08"},
            "category": {"type": "string", "description": f"조회할 지출 카테고리. {EXPENSE}"},
        },
        "required": ["month"],
    },
}

TOOLS = [
    add_transaction_tool,
    search_transactions_tool,
    update_transaction_tool,
    set_budget_tool,
    get_budget_status_tool,
]

print([t["name"] for t in TOOLS])
```

</details>

#### 5-3. 함수 연결과 시스템 프롬프트

<details open>
<summary>설명 펼치기</summary>

##### 이름 → 실제 함수 변환표

모델은 `"add_transaction"`이라는 **문자열**을 돌려준다. 그 문자열로 실제 함수를 찾아야 한다.

```python
TOOL_FUNCTIONS = {
    "add_transaction": manager.add_transaction,
    ...
}
```

**괄호를 붙이지 않는다.** 붙이면 지금 실행해서 그 결과를 담는 것이 된다.
`manager.`에 붙여 담으면 나중에 `**arguments`로 호출할 때 `self`가 자동으로 채워진다.

이 표에 없는 이름은 실행되지 않으므로 **안전장치 역할**도 한다.

##### 시스템 프롬프트

모델에게 매 요청마다 함께 보내는 **역할 설명서**다. 네 가지가 들어가야 한다.

**① 오늘 날짜** — 모델은 오늘이 며칠인지 모른다.

```python
today = datetime.now().strftime("%Y-%m-%d (%A)")
```

`strftime`은 `strptime`의 반대다. **f는 format** — datetime을 문자열로 바꾼다.
`%A`는 요일 이름이라 "지난 주말" 같은 표현도 처리할 수 있게 된다.

> 도구로 만들지 않는 이유: 항상 필요한 정보라서, 모델이 "물어봐야 하나?"를 판단할 필요조차 없게 만드는 편이 안전하다.

**② 카테고리 목록과 분류 힌트** — "점심"을 식비로, "택시"를 교통비로 분류하도록 예시를 준다.

**③ 검색 결과가 여러 건일 때의 규칙** — 명세서 요구사항이다.
*"검색 결과가 여러 개라면 임의로 수정하지 않고 특정할 조건을 안내한다."*
이건 **코드가 아니라 프롬프트로 구현되는 요구사항**이다.

**④ 도구를 반드시 쓰라는 지시** — 없으면 모델이 그럴듯한 답을 지어낸다.

</details>

In [ ]:
# 1) 이름 → 실제 함수 변환표를 만든다
#    왜  : 모델은 "add_transaction" 이라는 문자열만 돌려준다. 문자열로는 함수를 실행할 수 없으므로
#          "이 문자열이 오면 이 함수를 부른다"는 대응표가 필요하다.
#          이 표에 없는 이름은 실행되지 않으므로, 안전장치 역할도 한다.
#    방법: TOOL_FUNCTIONS = {"add_transaction": manager.add_transaction, ...}
#          ⚠️ 괄호를 붙이지 않는다. manager.add_transaction() 이라고 쓰면
#             지금 실행해서 그 결과값이 담긴다. 괄호 없이 써야 "함수 자체"가 담긴다.
#          manager. 에 붙여 담으면 나중에 **arguments 로 부를 때 self 가 자동으로 채워진다.



# 2) 오늘 날짜 문자열을 만든다
#    왜  : 모델은 오늘이 며칠인지 모른다. "오늘 점심 먹었어" 의 date 를 채우려면 알려줘야 한다.
#          도구로 만들지 않는 이유는, 항상 필요한 정보라서 모델이 "물어볼까?"를 판단할 필요조차
#          없게 만드는 편이 안전하기 때문이다.
#    방법: today = datetime.now().strftime("%Y-%m-%d (%A)")
#          strftime 은 strptime 의 반대다 (f = format). datetime 을 문자열로 바꾼다.
#          %A 는 요일 이름이라, "지난 주말" 같은 표현도 처리할 수 있게 된다.



# 3) SYSTEM_INSTRUCTION 을 f-string 으로 작성한다
#    왜  : 매 요청마다 함께 보내는 "역할 설명서"다. 대화 내용과 별개로 행동 규칙을 정해준다.
#          특히 아래 세 번째 항목은 코드가 아니라 프롬프트로만 구현되는 명세서 요구사항이다.
#    방법: 다음 네 가지를 반드시 넣는다.
#          ① 오늘 날짜  — 2)에서 만든 today 를 f-string 으로 끼워 넣는다
#          ② 카테고리 목록 + 분류 예시 — "점심·커피 → 식비", "택시·지하철 → 교통비" 처럼
#            예시를 주면 모델이 임의로 카테고리를 지어내는 일이 줄어든다
#          ③ 검색 결과가 2건 이상이면 임의로 고르지 말고 되물을 것
#            (명세서: "Agent가 임의로 수정하지 않고 특정할 조건을 안내한다")
#          ④ 추측하지 말고 반드시 도구로 확인할 것
#            이게 없으면 모델이 도구를 안 부르고 그럴듯한 답을 지어낸다
#          그 밖에 "금액은 천 단위 쉼표를 넣어 읽어줄 것" 같은 표현 규칙도 넣으면 좋다



# 4) 프롬프트를 출력해서 오늘 날짜가 제대로 들어갔는지 확인한다

##### 💡 함수 연결 + 시스템 프롬프트 — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
TOOL_FUNCTIONS = {
    "add_transaction": manager.add_transaction,
    "search_transactions": manager.search_transactions,
    "update_transaction": manager.update_transaction,
    "set_budget": manager.set_budget,
    "get_budget_status": manager.get_budget_status,
}

today = datetime.now().strftime("%Y-%m-%d (%A)")

SYSTEM_INSTRUCTION = f"""당신은 개인 예산 관리 비서입니다.

[오늘 날짜]
오늘은 {today}입니다.
'오늘', '어제', '이번 달', '지난주' 같은 표현은 이 날짜를 기준으로 계산하세요.

[카테고리]
지출: 식비, 교통비, 주거비, 문화생활, 기타
수입: 월급, 용돈, 부수입, 기타
사용자의 표현을 위 카테고리 중 하나로 알맞게 분류하세요.
(예: '점심'·'커피' -> 식비, '택시'·'지하철' -> 교통비, '영화' -> 문화생활)

[작업 규칙]
1. 추측하지 말고 반드시 도구를 사용해 확인하세요.
2. 거래를 수정하려면 거래 ID가 필요합니다.
   사용자가 ID를 모르면 먼저 search_transactions로 찾으세요.
3. 검색 결과가 2건 이상이면 절대 임의로 고르지 마세요.
   찾은 거래들을 보여주고, 어떤 것인지 특정할 조건(날짜, 금액, 설명)을 되물으세요.
4. 검색 결과가 0건이면 그 사실을 알리고 조건을 바꿔볼 것을 제안하세요.
5. 도구가 ok=false를 반환하면 message 내용을 알기 쉽게 설명하세요.
6. 금액은 12,000원처럼 천 단위 쉼표를 넣어 읽어주세요.
"""

print(SYSTEM_INSTRUCTION)
```

</details>

#### 5-4. 단발 호출 테스트

<details open>
<summary>설명 펼치기</summary>

Agent loop를 만들기 전에 **모델이 도구를 제대로 골라내는지**부터 확인한다.
루프까지 한꺼번에 만들면 문제가 생겼을 때 원인이 schema인지 루프인지 알 수 없다.

##### 요청 보내기

```python
interaction = client.interactions.create(
    model=model,
    input="오늘 점심으로 12000원 썼어",
    tools=TOOLS,
    system_instruction=SYSTEM_INSTRUCTION,
    store=False,
)
```

##### 응답 읽기

응답은 **여러 개의 step**으로 나뉘어 온다. 그중 `type`이 `"function_call"`인 것이 도구 호출 요청이다.

```python
for step in interaction.steps:
    if step.type == "function_call":
        print(step.name)         # 모델이 고른 도구 이름
        print(step.arguments)    # 모델이 만든 인자 (딕셔너리)
        print(step.id)           # 이 호출의 고유 ID
```

**이 시점에는 아직 함수가 실행되지 않았다.** 모델은 "불러줘"라고 요청만 한 상태다.

##### 확인할 것

- 모델이 `add_transaction`을 골랐는가
- `transaction_type`이 `"지출"`, `category`가 `"식비"`인가
- `date`에 **오늘 날짜**가 들어갔는가 → 시스템 프롬프트가 작동했다는 증거

##### 실제로 실행해보기

```python
tool_function = TOOL_FUNCTIONS[step.name]     # 이름으로 함수를 찾고
result = tool_function(**step.arguments)      # 실행한다
```

`**`는 `{"a": 1, "b": 2}` 딕셔너리를 `a=1, b=2` 형태로 **풀어서** 인자로 넣어준다.

</details>

In [ ]:
# 이 셀의 목적: Agent loop 를 만들기 전에 "모델이 도구를 제대로 골라내는지"만 먼저 확인한다.
# 루프까지 한꺼번에 만들면, 문제가 생겼을 때 원인이 schema 인지 루프인지 알 수 없다.


# 1) 요청을 보낸다
#    방법: interaction = client.interactions.create(
#              model=model,
#              input="오늘 점심으로 12000원 썼어",
#              tools=TOOLS,
#              system_instruction=SYSTEM_INSTRUCTION,
#              store=False,           # 이어갈 대화가 아니므로 저장하지 않는다
#          )



# 2) 응답에서 도구 호출 요청만 골라낸다
#    왜  : 응답은 여러 개의 step 으로 나뉘어 온다. 그중 type 이 "function_call" 인 것이
#          "이 함수를 이렇게 불러주세요" 라는 요청이다.
#          ⚠️ 이 시점에는 아직 함수가 실행되지 않았다. 모델은 요청만 한 상태다.
#    방법: for step in interaction.steps:
#              if step.type == "function_call":
#                  step.name       모델이 고른 도구 이름
#                  step.arguments  모델이 만든 인자 (딕셔너리)
#                  step.id         이 호출의 고유 ID (나중에 결과를 돌려줄 때 필요)
#    확인할 것:
#          - add_transaction 을 골랐는가
#          - transaction_type 이 "지출", category 가 "식비" 인가
#          - date 에 오늘 날짜가 들어갔는가  ← 시스템 프롬프트가 작동했다는 증거



# 3) 실제로 함수를 실행한다
#    왜  : 모델이 요청한 것을 우리가 대신 실행해준다. 실행 권한은 항상 우리 쪽에 있다.
#    방법: tool_function = TOOL_FUNCTIONS[step.name]     이름으로 함수를 찾고
#          result = tool_function(**step.arguments)      실행한다
#          ** 는 {"a": 1, "b": 2} 를 a=1, b=2 형태로 풀어서 인자로 넣어준다.



# 4) manager.transactions 의 마지막 항목을 출력해 실제로 등록됐는지 확인한다

##### 💡 단발 호출 테스트 — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
interaction = client.interactions.create(
    model=model,
    input="오늘 점심으로 12000원 썼어",
    tools=TOOLS,
    system_instruction=SYSTEM_INSTRUCTION,
    store=False,
)

for step in interaction.steps:
    if step.type == "function_call":
        print("도구 이름:", step.name)
        print("인자     :", step.arguments)
        print("호출 ID  :", step.id)

        tool_function = TOOL_FUNCTIONS[step.name]
        result = tool_function(**step.arguments)
        print("실행 결과:", result)

print()
print(manager.transactions[-1])
```

</details>

### Phase 6 — Agent loop

<details open>
<summary>설명 펼치기</summary>

단발 호출은 **도구를 한 번 부르는 것까지**만 처리한다.
하지만 *"8월 27일 카페에서 쓴 금액을 4500원으로 수정해줘"* 는 두 단계가 필요하다.

```
1. search_transactions 로 거래를 찾아 ID 확인
2. 그 ID로 update_transaction 호출
```

모델은 **첫 번째 결과를 받아본 뒤에야** 두 번째 호출을 만들 수 있다.
그래서 "도구 요청이 더 없을 때까지 반복"하는 구조가 필요하다.

##### 최대 반복 횟수를 반드시 둔다

명세서 요구사항이다. 없으면 모델이 계속 도구를 부를 때 **비용과 시간이 무한히 늘어난다.**
이 프로젝트에서 가장 긴 작업이 2단계이므로 `max_turns=5`면 충분하다.

##### `execute_tool_call` — 도구를 안전하게 실행

모델이 보낸 인자가 우리 함수와 안 맞을 수 있다. 그대로 실행하면 **Agent 전체가 죽는다.**

```python
try:
    return tool_function(**step.arguments)
except TypeError as error:
    return {"ok": False, "error": "INVALID_ARGUMENTS", "message": f"잘못된 인자: {error}"}
except Exception as error:
    return {"ok": False, "error": "TOOL_FAILED", "message": f"도구 실행 실패: {type(error).__name__}"}
```

> `Exception`은 거의 모든 에러의 부모다. 위에 두면 `TypeError`도 여기서 잡혀버린다.
> **구체적인 것을 위에, 포괄적인 것을 아래에** 두는 게 규칙이다.

등록 안 된 이름은 `TOOL_FUNCTIONS.get(step.name)`으로 막는다.
`[...]`로 꺼내면 없는 이름일 때 `KeyError`로 죽는다.

**모든 실패가 규약대로 `{"ok": False, ...}`로 나오는 것이 핵심이다.**

##### 루프의 구조

```
반복 (최대 max_turns번):
    1. 모델에 요청을 보낸다
    2. 응답에서 function_call step들을 골라낸다
    3. 하나도 없으면 → 최종 답변이다. 반환하고 끝
    4. 있으면 → 전부 실행하고, 결과를 모아 다음 입력으로 만든다
    5. previous_interaction_id 를 기록해 대화를 이어간다
```

##### 대화를 이어가는 방법

| 부분 | 의미 |
|---|---|
| `store=True` | 대화 상태를 서버에 저장한다. 이어가려면 필요하다 |
| `previous_interaction_id` | **직전 요청의 id.** 이걸 줘야 모델이 앞 대화를 기억한다 |
| 첫 턴 | 이전 대화가 없으므로 이 항목을 **아예 넣지 않는다** |

그래서 요청을 딕셔너리로 조립한 뒤 `client.interactions.create(**request)` 로 푼다.
조건에 따라 항목을 넣거나 뺄 수 있기 때문이다.

##### 결과를 모델에게 돌려주는 형태

```python
{
    "type": "function_result",
    "name": step.name,
    "call_id": step.id,                                    # ⚠️ 모델이 보낸 id와 같아야 한다
    "result": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}],
}
```

`call_id`가 어긋나면 모델이 어느 호출의 결과인지 알 수 없다.
결과는 문자열로 바꿔 보낸다 → `json.dumps` (파일이 아니라 문자열이므로 `s`가 붙은 쪽)

##### 콘솔 출력 (명세서 필수 기능)

*"Agent가 호출한 도구와 실행 결과를 콘솔에서 확인"* 이 필수다.
이 출력이 **연속 도구 호출을 시연하는 증거**가 되므로 신경 써서 만든다.

```
🔧 [도구 호출] search_transactions
   인자: {'date': '2026-08-27', 'keyword': '카페'}
✅ [실행 결과] 조건에 맞는 거래 1건을 찾았습니다.
```

</details>

In [ ]:
# 1) execute_tool_call(step) — 도구 하나를 안전하게 실행하는 함수
#    왜  : 모델이 보낸 인자가 우리 함수와 안 맞을 수 있다. 그대로 실행하면 예외가 나면서
#          Agent 전체가 죽는다. 어떤 실패든 규약대로 {"ok": False, ...} 로 바꿔 돌려주면,
#          루프는 결과를 한 가지 방법으로만 다루면 된다.
#    방법: tool_function = TOOL_FUNCTIONS.get(step.name)
#            .get 을 쓰는 이유: 모델이 없는 이름을 보낼 수 있다.
#            TOOL_FUNCTIONS[step.name] 로 꺼내면 그때 KeyError 로 죽는다.
#          None 이면 UNKNOWN_TOOL 반환
#          try: return tool_function(**step.arguments)
#          except TypeError  → INVALID_ARGUMENTS   (인자 개수나 이름이 안 맞을 때)
#          except Exception  → TOOL_FAILED         (그 밖의 모든 에러)
#          ⚠️ Exception 은 거의 모든 에러의 부모라서 위에 두면 TypeError 도 여기서 잡힌다.
#             구체적인 것을 위에, 포괄적인 것을 아래에 둔다.



# 2) 콘솔 출력 함수 두 개
#    왜  : 명세서 필수 기능 — "Agent가 호출한 도구와 실행 결과를 콘솔에서 확인".
#          이 출력이 "검색 → 수정 연속 호출"을 시연하는 증거가 되므로 보기 좋게 만든다.
#    방법: print_tool_call(step)    → 도구 이름과 인자를 출력
#          print_tool_result(result) → 성공이면 ✅, 실패면 ❌ 와 error 코드를 함께 출력



# 3) run_agent(user_input, max_turns=5) — Agent loop
#    왜  : "8월 27일 카페 금액을 4500원으로" 같은 요청은 검색 → 수정 두 단계가 필요하다.
#          모델은 첫 번째 결과를 받아본 뒤에야 두 번째 호출을 만들 수 있으므로,
#          "도구 요청이 더 없을 때까지" 반복해야 한다.
#          최대 횟수를 두는 이유: 없으면 모델이 계속 도구를 부를 때 비용과 시간이 무한히 늘어난다.
#    방법: next_input = user_input           첫 턴은 사용자 문장, 두 번째부터는 도구 결과가 입력이다
#          previous_interaction_id = None    첫 턴에는 이전 대화가 없다
#          logs = []                          도구 호출 기록
#
#          for turn in range(1, max_turns + 1):
#              ① 요청을 딕셔너리로 조립한다
#                 model / input / tools / system_instruction / store=True
#                 previous_interaction_id 는 첫 턴에만 없으므로, if 로 있을 때만 넣는다
#                 → 그래서 딕셔너리로 만들었다가 create(**request) 로 푼다
#              ② function_calls = [s for s in interaction.steps if s.type == "function_call"]
#              ③ 비어 있으면 최종 답변이다 → interaction.output_text 를 담아 return
#              ④ 있으면 전부 실행하고, 결과를 다음 입력으로 만든다
#                 next_input 에 아래 형태의 딕셔너리를 하나씩 담는다
#                   {"type": "function_result",
#                    "name": step.name,
#                    "call_id": step.id,      ⚠️ 모델이 보낸 id 와 같아야 한다. 어긋나면
#                                                모델이 어느 호출의 결과인지 알 수 없다
#                    "result": [{"type": "text",
#                                "text": json.dumps(result, ensure_ascii=False)}]}
#                    결과는 문자열로 바꿔 보낸다 → 파일이 아니라 문자열이므로 s 가 붙은 dumps
#              ⑤ previous_interaction_id = interaction.id   다음 턴에서 대화를 이어가려고 기억해 둔다
#
#          for 가 끝까지 돌았다 = 한도를 다 썼는데도 안 끝났다 → 실패로 마무리한다

##### 💡 Agent loop — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def execute_tool_call(step):
    """도구 하나를 안전하게 실행한다."""
    tool_function = TOOL_FUNCTIONS.get(step.name)
    if tool_function is None:
        return {"ok": False, "error": "UNKNOWN_TOOL",
                "message": f"허용되지 않은 도구입니다: {step.name}"}

    try:
        return tool_function(**step.arguments)
    except TypeError as error:
        return {"ok": False, "error": "INVALID_ARGUMENTS",
                "message": f"잘못된 인자: {error}"}
    except Exception as error:
        return {"ok": False, "error": "TOOL_FAILED",
                "message": f"도구 실행 실패: {type(error).__name__}"}


def print_tool_call(step):
    print(f"🔧 [도구 호출] {step.name}")
    print(f"   인자: {step.arguments}")


def print_tool_result(result):
    mark = "✅" if result.get("ok") else "❌"
    detail = result.get("message", "")
    if not result.get("ok"):
        detail = f"({result.get('error')}) {detail}"
    print(f"{mark} [실행 결과] {detail}\n")


def run_agent(user_input, max_turns=5):
    next_input = user_input
    previous_interaction_id = None
    logs = []

    print("─" * 50)

    for turn in range(1, max_turns + 1):
        request = {
            "model": model,
            "input": next_input,
            "tools": TOOLS,
            "system_instruction": SYSTEM_INSTRUCTION,
            "store": True,
        }
        if previous_interaction_id is not None:
            request["previous_interaction_id"] = previous_interaction_id

        interaction = client.interactions.create(**request)
        function_calls = [s for s in interaction.steps if s.type == "function_call"]

        # 도구 요청이 없으면 최종 답변이다
        if not function_calls:
            print("─" * 50)
            print(f"💬 {interaction.output_text}")
            return {"ok": True, "answer": interaction.output_text,
                    "turns": turn, "tool_logs": logs}

        next_input = []
        for step in function_calls:
            print_tool_call(step)
            result = execute_tool_call(step)
            print_tool_result(result)

            logs.append({"turn": turn, "tool": step.name,
                         "arguments": step.arguments, "result": result})
            next_input.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": [{"type": "text",
                            "text": json.dumps(result, ensure_ascii=False)}],
            })

        previous_interaction_id = interaction.id

    print("─" * 50)
    return {"ok": False, "answer": None, "turns": max_turns,
            "tool_logs": logs, "error": "최대 반복 횟수를 초과했습니다."}
```

</details>

In [ ]:
# Agent loop 테스트 — 도구 한 번이면 끝나는 요청부터

result = run_agent("오늘 커피 4500원 썼어")

print()
print("호출 순서:", [log["tool"] for log in result["tool_logs"]])

### Phase 7 — 필수 시연 시나리오 검증

<details open>
<summary>설명 펼치기</summary>

명세서가 요구하는 다섯 가지를 순서대로 확인한다.
**여기까지 통과하면 "작동하는 완성품"이다.**

각 시나리오는 콘솔에 도구 호출이 보이는 상태로 실행하고 **화면을 캡처해 둔다.** README에 넣을 자료가 된다.

| # | 시나리오 | 확인할 것 |
|---|---|---|
| 1 | 한 문장으로 지출 등록 | `add_transaction` 1회, 오늘 날짜가 자동으로 들어갔는가 |
| 2 | 자연어 조건 검색 | `search_transactions` 1회, 기간·카테고리가 제대로 변환됐는가 |
| 3 | **검색 후 수정** | `search` → `update` **2회 연속** 호출이 콘솔에 찍히는가 |
| 4 | 예산 설정 및 조회 | `set_budget` → `get_budget_status` |
| 5 | 없는 거래 수정 요청 | `NOT_FOUND`, **프로그램이 죽지 않고 대화가 이어지는가** |

##### ⚠️ 시나리오 3 준비

명세서: *"연속 도구 호출 시연에서는 검색 조건과 일치하는 거래가 하나인 경우를 사용한다."*

8월 27일 카페 거래가 **정확히 1건**만 있는 상태를 만들어 두고 실행한다.
여러 건이면 모델이 되묻기 때문에 연속 호출이 일어나지 않는다.

```python
print(manager.search_transactions(date="2026-08-27", keyword="카페")["data"]["count"])   # 1이어야 한다
```

</details>

In [ ]:
# 시나리오 1 — 한 문장으로 지출 등록
run_agent("오늘 점심으로 12000원 썼어")

In [ ]:
# 시나리오 2 — 자연어 조건 검색
run_agent("8월에 식비로 쓴 거 보여줘")

In [ ]:
# 시나리오 3 — 검색 후 수정 (⭐ 연속 도구 호출)
# 먼저 조건에 맞는 거래가 1건인지 확인한다
print("일치 건수:", manager.search_transactions(date="2026-08-27", keyword="카페")["data"]["count"])
print()

result = run_agent("2026년 8월 27일 카페에서 쓴 금액을 4500원으로 수정해줘")
print()
print("호출 순서:", [log["tool"] for log in result["tool_logs"]])
# ['search_transactions', 'update_transaction'] 이면 성공

In [ ]:
# 시나리오 4 — 예산 설정 및 조회
run_agent("이번 달 식비 예산을 30만원으로 설정해줘")
print()
run_agent("이번 달 식비 얼마나 남았어?")

In [ ]:
# 시나리오 5 — 존재하지 않는 거래 수정 (오류 처리)
run_agent("999번 거래 금액을 5000원으로 바꿔줘")
# NOT_FOUND가 찍히고, 프로그램이 죽지 않고 안내 문장이 나와야 한다

In [ ]:
# 추가 확인 — 검색 결과가 여러 건일 때 되묻는지 (명세서 요구사항)
# 준비: 카페 거래를 2건 이상 만든다
manager.add_transaction("지출", "식비", 6000, "2026-08-26", "카페 라떼")

run_agent("카페에서 쓴 거 수정해줘")
# 임의로 고치지 않고, 목록을 보여주며 날짜 등을 되물어야 한다

### Phase 8 — 선택 기능

<details open>
<summary>설명 펼치기</summary>

필수 기능이 전부 통과한 뒤에 붙인다. **1차 완성품을 깨뜨리지 않는 것이 원칙이다.**

#### 8-1. `delete_transaction` — 거래 삭제

`update_transaction`과 구조가 거의 같다. ID로 찾고, 없으면 `NOT_FOUND`, 있으면 목록에서 뺀다.

```python
self.transactions.remove(target)      # 그 값을 목록에서 제거
```

**지우기 전에 복사해 둔다.** 삭제된 내용을 `data`에 담아야 모델이 "무엇을 지웠는지" 알려줄 수 있는데,
지우고 나면 확인할 방법이 없기 때문이다.

> `next_id`는 되돌리지 않는다. 지운 번호를 재사용하면 예전 대화에 나온 ID가 다른 거래를 가리키게 된다.

만든 뒤에는 **schema도 만들어 `TOOLS`와 `TOOL_FUNCTIONS`에 추가**해야 모델이 쓸 수 있다.

</details>

In [ ]:
def delete_transaction(self, transaction_id):

    # 1) ID를 정수로 바꾼다 (update_transaction 과 같은 이유)
    #    방법: try / except (TypeError, ValueError) → NOT_FOUND return



    # 2) 그 ID의 거래를 찾는다. 없으면 NOT_FOUND
    #    방법: update_transaction 의 2) 와 같은 for 문



    # 3) 지우기 전에 복사해 둔다  ⭐ 순서가 중요하다
    #    왜  : 삭제한 내용을 data 에 담아야 모델이 "무엇을 지웠는지" 사용자에게 알려줄 수 있다.
    #          그런데 지우고 나면 확인할 방법이 없으므로, 반드시 제거하기 전에 복사해야 한다.
    #    방법: removed = target.copy()



    # 4) 목록에서 제거하고 저장한다
    #    왜  : next_id 는 되돌리지 않는다. 지운 번호를 재사용하면
    #          예전 대화에 나온 ID가 다른 거래를 가리키게 된다.
    #    방법: self.transactions.remove(target) 그다음 self._save()



    # 5) 삭제된 거래를 담아 성공 반환



BudgetManager.delete_transaction = delete_transaction


# ---- schema 만들고 등록하기 ----
#    왜  : 함수를 만들기만 하면 모델은 그런 도구가 있는 줄 모른다.
#          schema 를 만들어 TOOLS 에 넣고, TOOL_FUNCTIONS 에도 이름을 등록해야 쓸 수 있다.
#    방법: delete_transaction_tool = {...}
#          TOOLS.append(delete_transaction_tool)
#          TOOL_FUNCTIONS["delete_transaction"] = manager.delete_transaction


# ---- 확인 ----
# print(manager.delete_transaction(1)["message"])
# print(manager.delete_transaction(999))          # NOT_FOUND

##### 💡 delete_transaction — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def delete_transaction(self, transaction_id):
    try:
        transaction_id = int(transaction_id)
    except (TypeError, ValueError):
        return {"ok": False, "error": "NOT_FOUND",
                "message": f"거래 ID가 올바르지 않습니다: {transaction_id}"}

    target = None
    for t in self.transactions:
        if t["id"] == transaction_id:
            target = t
            break

    if target is None:
        return {"ok": False, "error": "NOT_FOUND",
                "message": f"{transaction_id}번 거래를 찾을 수 없습니다."}

    removed = target.copy()          # 지우기 전에 복사
    self.transactions.remove(target)
    self._save()

    return {
        "ok": True,
        "data": removed,
        "message": (f"[{removed['id']}] {removed['date']} {removed['category']} "
                    f"{removed['amount']:,}원 거래를 삭제했습니다."),
    }


BudgetManager.delete_transaction = delete_transaction

delete_transaction_tool = {
    "type": "function",
    "name": "delete_transaction",
    "description": ("거래 ID로 거래를 삭제합니다. "
                    "ID를 모르면 먼저 search_transactions로 찾아 확인한 뒤 호출하세요."),
    "parameters": {
        "type": "object",
        "properties": {
            "transaction_id": {"type": "integer", "description": "삭제할 거래의 ID"},
        },
        "required": ["transaction_id"],
    },
}

TOOLS.append(delete_transaction_tool)
TOOL_FUNCTIONS["delete_transaction"] = manager.delete_transaction
```

</details>

#### 8-2. 로그 파일

<details open>
<summary>설명 펼치기</summary>

콘솔 출력과 같은 내용을 파일에도 남긴다. 이미 출력 함수가 있으므로 **파일에 한 줄 덧붙이는 것**만 추가하면 된다.

```python
with open("agent.log", "a", encoding="utf-8") as f:
    f.write("...\n")
```

`"a"`는 **append(덧붙이기)** 모드다. `"w"`로 열면 매번 기존 내용을 지워버린다.

시각은 `datetime.now().strftime("%Y-%m-%d %H:%M:%S")` 로 만든다.

만든 뒤 `run_agent` 안의 도구 실행 자리에서 이 함수를 불러주면 된다.

</details>

In [ ]:
def write_log(turn, step, result):

    # 1) 현재 시각 문자열을 만든다
    #    왜  : 로그는 "언제 무슨 일이 있었는지"를 남기는 것이라 시각이 없으면 쓸모가 크게 준다.
    #    방법: datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    #          %H 시 / %M 분 / %S 초



    # 2) 파일에 한 줄 덧붙인다
    #    왜  : "a"(append) 모드로 열어야 기존 내용 뒤에 이어 쓴다.
    #          "w" 로 열면 실행할 때마다 앞의 기록이 전부 지워진다.
    #    방법: with open("agent.log", "a", encoding="utf-8") as f:
    #              f.write(한 줄 + "\n")
    #          한 줄 형식 예:
    #          2026-08-28 14:03:11 | turn=1 | search_transactions | {"date": "..."} | ok=True
    #          인자는 json.dumps(step.arguments, ensure_ascii=False) 로 문자열로 만든다



# 만든 뒤: run_agent 안의 print_tool_result(result) 다음 줄에
#          write_log(turn, step, result) 를 추가하고 run_agent 셀을 다시 실행한다.

# run_agent("오늘 택시비 8000원 썼어")

# with open("agent.log", "r", encoding="utf-8") as f:
#     print(f.read())

##### 💡 write_log — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def write_log(turn, step, result):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = (f"{now} | turn={turn} | {step.name} | "
            f"{json.dumps(step.arguments, ensure_ascii=False)} | ok={result.get('ok')}\n")
    with open("agent.log", "a", encoding="utf-8") as f:
        f.write(line)


# run_agent 안에서 이렇게 부른다
#     print_tool_result(result)
#     write_log(turn, step, result)      # ← 이 줄 추가
```

</details>

#### 8-3. 월별 Markdown 보고서

<details open>
<summary>설명 펼치기</summary>

`generate_monthly_report(month)` 를 만든다.

```
1. 그 달 거래를 모은다          → 없으면 NO_DATA
2. 총수입 · 총지출 · 잔액 계산
3. 예산 현황 표 (get_budget_status 를 재사용할 수 있다)
4. 거래 내역 표
5. report_2026-08.md 로 저장
```

##### Markdown 표를 문자열로 만들기

```python
lines = []
lines.append("| ID | 날짜 | 금액 |")
lines.append("|---|---|---|")
for t in 거래목록:
    lines.append(f"| {t['id']} | {t['date']} | {t['amount']:,} |")

content = "\n".join(lines)
```

리스트에 한 줄씩 담았다가 `"\n".join(...)` 으로 합치는 방식이,
문자열을 계속 `+`로 이어붙이는 것보다 읽기 쉽고 빠르다.

이것도 도구로 등록하면 *"8월 지출 내역을 파일로 저장해줘"* 가 동작한다.

</details>

In [ ]:
def generate_monthly_report(self, month):

    # 1) 월 형식을 검사한다
    #    방법: self._validate_month(month) 가 False면 INVALID_MONTH return



    # 2) 그 달 거래를 모은다
    #    왜  : 거래가 하나도 없으면 만들 보고서가 없다. 빈 파일을 만드는 것보다
    #          "그 달엔 내역이 없다"고 알려주는 편이 낫다.
    #    방법: t["date"].startswith(month) 인 거래만 모으고, 0건이면 NO_DATA return
    #          보기 좋게 날짜순으로 정렬해 둔다



    # 3) 총수입 / 총지출 / 잔액을 계산한다
    #    왜  : 보고서 맨 위에 요약이 있어야 한눈에 파악된다.
    #    방법: type 이 "수입" 인 것들의 amount 합, "지출" 인 것들의 amount 합
    #          sum(t["amount"] for t in 목록 if 조건) 형태로 한 줄에 쓸 수 있다



    # 4) 마크다운 문자열을 만든다
    #    왜  : 문자열을 + 로 계속 이어붙이면 읽기 어렵고 느리다.
    #          리스트에 한 줄씩 담았다가 마지막에 합치는 방식이 낫다.
    #    방법: lines = [] 에 한 줄씩 append 하고, 마지막에 "\n".join(lines)
    #          구성: 제목 → 요약 → 예산 현황 표 → 거래 내역 표
    #          예산 현황은 self.get_budget_status(month) 를 그대로 재사용할 수 있다
    #          (이미 만들어 둔 기능을 다시 쓰는 것이 같은 계산을 또 짜는 것보다 낫다)
    #          표는 이런 모양이다
    #            | ID | 날짜 | 금액 |
    #            |---|---|---|
    #            | 1 | 2026-08-27 | 12,000 |



    # 5) 파일로 저장하고 결과를 돌려준다
    #    방법: filename = f"report_{month}.md"
    #          with open(filename, "w", encoding="utf-8") as f: f.write(내용)
    #          data 에는 파일명과 요약(건수/수입/지출/잔액)을 담아준다



BudgetManager.generate_monthly_report = generate_monthly_report


# ---- 확인 ----
# print(manager.generate_monthly_report("2026-08")["message"])
# with open("report_2026-08.md", "r", encoding="utf-8") as f:
#     print(f.read())

##### 💡 generate_monthly_report — 정답

<details>
<summary>막혔을 때만 펼쳐보기</summary>

```python
def generate_monthly_report(self, month):
    # 1) 월 형식
    if not self._validate_month(month):
        return {"ok": False, "error": "INVALID_MONTH",
                "message": "월은 YYYY-MM 형식이어야 합니다. 예: 2026-08"}

    # 2) 그 달 거래 모으기
    targets = [t for t in self.transactions if t["date"].startswith(month)]
    if not targets:
        return {"ok": False, "error": "NO_DATA",
                "message": f"{month}에 거래 내역이 없습니다."}

    targets.sort(key=lambda t: (t["date"], t["id"]))

    # 3) 합계
    income = sum(t["amount"] for t in targets if t["type"] == "수입")
    expense = sum(t["amount"] for t in targets if t["type"] == "지출")

    # 4) 마크다운 만들기
    year, mon = month.split("-")
    lines = [f"# {year}년 {mon}월 거래 보고서", ""]

    lines += ["## 요약", "",
              f"- 총 수입: {income:,}원",
              f"- 총 지출: {expense:,}원",
              f"- 잔액: {income - expense:,}원", ""]

    status = self.get_budget_status(month)
    if status["ok"]:
        lines += ["## 예산 현황", "",
                  "| 카테고리 | 예산 | 사용 | 남음 | 사용률 |",
                  "|---|---|---|---|---|"]
        for item in status["data"]["items"]:
            lines.append(f"| {item['category']} | {item['budget']:,} | {item['spent']:,} | "
                         f"{item['remaining']:,} | {item['usage_rate']}% |")
        lines.append("")

    lines += ["## 거래 내역", "",
              "| ID | 날짜 | 유형 | 카테고리 | 금액 | 설명 |",
              "|---|---|---|---|---|---|"]
    for t in targets:
        lines.append(f"| {t['id']} | {t['date']} | {t['type']} | {t['category']} | "
                     f"{t['amount']:,} | {t['description']} |")

    content = "\n".join(lines)

    # 5) 파일 저장
    filename = f"report_{month}.md"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

    return {
        "ok": True,
        "data": {"filename": filename, "count": len(targets),
                 "income": income, "expense": expense, "balance": income - expense},
        "message": (f"{month} 보고서를 {filename} 파일로 저장했습니다. "
                    f"거래 {len(targets)}건, 수입 {income:,}원, 지출 {expense:,}원"),
    }


BudgetManager.generate_monthly_report = generate_monthly_report
```

</details>

### Phase 9 — `.py` 파일로 옮기기

<details open>
<summary>설명 펼치기</summary>

---

여기까지 노트북에서 전부 동작하는 것을 확인한 뒤에 시작한다.

| 옮길 것 | 어디로 |
|---|---|
| `BudgetManager` 클래스 전체 | `budget_manager.py` |
| 도구 schema, `TOOL_FUNCTIONS`, `SYSTEM_INSTRUCTION`, `execute_tool_call`, `run_agent` | `agent.py` |

##### 옮긴 뒤에 할 일

**노트북의 해당 정의 셀은 지우고 `import`로 바꾼다.** 같은 코드를 두 곳에 남겨두면 어느 쪽이 최신인지 알 수 없게 된다.

```python
%load_ext autoreload
%autoreload 2                       # .py를 고치면 자동으로 다시 불러온다

from budget_manager import BudgetManager
```

`%autoreload 2`가 없으면 `.py`를 고쳐도 노트북이 **처음 불러온 옛 코드를 계속 쓴다.** "고쳤는데 왜 그대로지?"의 대부분이 이것 때문이다.

##### `agent.py`에 추가할 것 — 대화 실행부

```python
def main():
    ...
    while True:
        user_input = input("\n나: ").strip()
        if user_input in ("종료", "exit", "quit"):
            break
        ...

if __name__ == "__main__":
    main()
```

`if __name__ == "__main__":`은 **"이 파일을 직접 실행했을 때만"**이라는 뜻이다. 다른 파일에서 `import`할 때는 실행되지 않는다. 이게 없으면 `import agent` 하는 순간 대화창이 켜져버린다.

##### 확인

터미널에서 `python agent.py`를 실행하고, **시나리오 5개를 다시 한 번 통과**시킨다. 노트북에서 됐다고 `.py`에서도 되는 건 아니다.

##### ⚠️ 옮긴 뒤 주의

노트북과 `python agent.py`를 **동시에 실행하면** 둘 다 `data.json` 전체를 덮어써서, 나중에 저장한 쪽이 상대의 변경분을 지운다. 동시에 쓸 일이 있으면 노트북 쪽만 파일을 분리한다.

```python
manager = BudgetManager(data_file="test_data.json")
```

---

</details>

### Phase 10 — 마무리

<details open>
<summary>설명 펼치기</summary>

---

- `.gitignore` 확인 — **`.env`가 들어있는지 반드시 확인한 뒤 첫 커밋**을 한다. 한 번 올라간 키는 나중에 파일을 지워도 커밋 기록에 남는다
- `README.md` 작성 — 기획서 14절의 구성안 참고
- 실행 화면 캡처 — 특히 **연속 도구 호출** 장면
- 기획서 12절 체크리스트로 빠진 것 최종 점검

</details>